In [ ]:
import pandas as pd 
import numpy as np

# Pulling SFT + LORA model done by bhavana from hugging face to properly evaluate the model results and see if this step is working fine

In [2]:
# Cell 2: Test
import sys
sys.path.insert(0, '/home/bsanghi2/.local/lib/python3.11/site-packages')

import torch
import torchvision
from transformers import PreTrainedModel, AutoModel
from peft import get_peft_model, LoraConfig, TaskType

print(f"✓ Torch: {torch.__version__}")
print(f"✓ Torchvision: {torchvision.__version__}")
print("✓ Transformers: WORKING")
print("✓ PEFT: WORKING")
print("\n🎉 DONE!")

✓ Torch: 2.4.1+cu121
✓ Torchvision: 0.19.1+cu121
✓ Transformers: WORKING
✓ PEFT: WORKING

🎉 DONE!


In [3]:
# Cell 1: Load model with cleaned config
import sys
sys.path.insert(0, '/home/bsanghi2/.local/lib/python3.11/site-packages')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig
import json
from huggingface_hub import hf_hub_download
import inspect

BASE = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER = "bhavanasanghi9/clarifier-sft-lora"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
)

print("Downloading and fixing adapter config...")
config_path = hf_hub_download(repo_id=ADAPTER, filename="adapter_config.json")
with open(config_path, 'r') as f:
    config_dict = json.load(f)

# Get valid parameters for LoraConfig
valid_params = set(inspect.signature(LoraConfig.__init__).parameters.keys())

# Filter config to only include valid parameters
cleaned_config = {k: v for k, v in config_dict.items() if k in valid_params}

print(f"Removed incompatible parameters: {set(config_dict.keys()) - set(cleaned_config.keys())}")

# Create config from cleaned dict
config = LoraConfig(**cleaned_config)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER, config=config)

print("✓ Model loaded successfully!")

Loading tokenizer...
Loading base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
# Cell 1: Load evaluation data from AmbigNQ
import sys
sys.path.insert(0, '/home/bsanghi2/.local/lib/python3.11/site-packages')

import pandas as pd
from datasets import load_dataset

# Load the original AmbigNQ validation set
print("Loading AmbigNQ validation set...")
ds = load_dataset("sewon/ambig_qa", "light")
eval_data = ds['validation']

print(f"Loaded {len(eval_data)} validation examples")
print(f"Sample keys: {eval_data[0].keys()}")

Loading AmbigNQ validation set...
Loaded 2002 validation examples
Sample keys: dict_keys(['id', 'question', 'annotations'])


In [8]:
# Cell: CORRECT inference function matching training format
def generate_response_correct(question, max_new_tokens=512, temperature=0.7):
    """Generate response using the SAME format as training"""
    
    system_msg = (
        "You are a question understanding agent trained on AmbigQA-style ambiguity. "
        "For each user question:\n"
        "1) Decide if it is ambiguous (i.e., multiple plausible interpretations/answers).\n"
        "2) Explain your reasoning.\n"
        "3) List facets if ambiguous.\n"
        "4) Ask a clarifying question OR give a direct answer.\n\n"
        "Format:\n"
        "Action: Clarify|Answer\n"
        "Reasoning: ...\n"
        "Facets: [...]\n"
        "Response: ...\n"
    )
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": question}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the assistant's response
    if "assistant\n" in response:
        response = response.split("assistant\n")[-1]
    elif "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "")
    
    return response.strip()

# Test it
test_q = "When did the simpsons first air on television?"
response = generate_response_correct(test_q, temperature=0.1)
print(f"Question: {test_q}")
print(f"\nModel output:\n{response}")

Question: When did the simpsons first air on television?

Model output:
Action: Answer
Reasoning: The question is not ambiguous because it specifies the show "The Simpsons" and asks about its television debut date, which is a unique and well-documented event. Therefore, the question contains enough details to determine the exact answer.
Facets: []
Response: December 15, 1989


In [13]:
# Cell: Complete evaluation with CORRECT ambiguity detection
import re
import pandas as pd
from tqdm import tqdm

def classify_ambiguity(example):
    """Determine if a question is ambiguous based on AmbigNQ annotations"""
    if 'annotations' in example and example['annotations']:
        annotations = example['annotations']
        if 'type' in annotations:
            # type is a list, check first element
            ann_type = annotations['type'][0] if isinstance(annotations['type'], list) else annotations['type']
            # 'multipleQAs' = ambiguous, 'singleAnswer' = not ambiguous
            return ann_type == 'multipleQAs'
    return False

def parse_model_output(output):
    """Parse the Action/Reasoning/Facets/Response format"""
    action = None
    reasoning = None
    facets = []
    response = None
    
    # Extract Action
    action_match = re.search(r'Action:\s*(Clarify|Answer)', output, re.IGNORECASE)
    if action_match:
        action = action_match.group(1)
    
    # Extract Reasoning
    reasoning_match = re.search(r'Reasoning:\s*(.+?)(?=\nFacets:|$)', output, re.DOTALL)
    if reasoning_match:
        reasoning = reasoning_match.group(1).strip()
    
    # Extract Facets
    facets_match = re.search(r'Facets:\s*\[(.+?)\]', output)
    if facets_match:
        facets = [f.strip().strip("'\"") for f in facets_match.group(1).split(',') if f.strip()]
    
    # Extract Response
    response_match = re.search(r'Response:\s*(.+?)$', output, re.DOTALL)
    if response_match:
        response = response_match.group(1).strip()
    
    return {
        'action': action,
        'reasoning': reasoning,
        'facets': facets,
        'response': response
    }

def evaluate_with_correct_format(num_samples=100):
    """Evaluate using the training format"""
    results = []
    
    print(f"Evaluating {num_samples} examples...")
    for i in tqdm(range(min(num_samples, len(eval_data)))):
        example = eval_data[i]
        question = example['question']
        is_ambiguous = classify_ambiguity(example)
        
        # Generate with correct format
        raw_output = generate_response_correct(question, temperature=0.1)
        parsed = parse_model_output(raw_output)
        
        # Check if correct
        predicted_action = parsed['action']
        
        if is_ambiguous:
            correct = (predicted_action == 'Clarify')
        else:
            correct = (predicted_action == 'Answer')
        
        results.append({
            'question': question,
            'is_ambiguous': is_ambiguous,
            'predicted_action': predicted_action,
            'response': parsed['response'],
            'facets': parsed['facets'],
            'correct': correct,
            'raw_output': raw_output
        })
    
    return pd.DataFrame(results)

# Run evaluation
print("Running evaluation with CORRECT format...")
results_df = evaluate_with_correct_format(100)

# Calculate metrics
print("\n" + "="*80)
print("EVALUATION RESULTS (CORRECT FORMAT)")
print("="*80)

accuracy = results_df['correct'].mean()
print(f"\n🎯 Overall Accuracy: {accuracy:.2%}")

ambig_examples = results_df[results_df['is_ambiguous'] == True]
unambig_examples = results_df[results_df['is_ambiguous'] == False]

if len(ambig_examples) > 0:
    ambig_acc = ambig_examples['correct'].mean()
    print(f"\n🔍 Ambiguous questions:")
    print(f"   Total: {len(ambig_examples)}")
    print(f"   Accuracy: {ambig_acc:.2%}")

if len(unambig_examples) > 0:
    unambig_acc = unambig_examples['correct'].mean()
    print(f"\n✅ Unambiguous questions:")
    print(f"   Total: {len(unambig_examples)}")
    print(f"   Accuracy: {unambig_acc:.2%}")

print("\n" + "="*80)

# Save results
results_df.to_csv('/mnt/user-data/outputs/sft_evaluation_corrected.csv', index=False)
print("\n✓ Results saved!")

Running evaluation with CORRECT format...
Evaluating 100 examples...


100%|██████████| 100/100 [05:08<00:00,  3.08s/it]



EVALUATION RESULTS (CORRECT FORMAT)

🎯 Overall Accuracy: 38.00%

🔍 Ambiguous questions:
   Total: 61
   Accuracy: 3.28%

✅ Unambiguous questions:
   Total: 39
   Accuracy: 92.31%



OSError: Cannot save file into a non-existent directory: '/mnt/user-data/outputs'

In [15]:
# Cell: Debug - let's see what the model is actually outputting
print("DEBUGGING: Let's see actual model outputs\n")
print("="*80)

# Check a few ambiguous examples
ambig_examples = results_df[results_df['is_ambiguous'] == True].head(5)

for idx, row in ambig_examples.iterrows():
    print(f"\nQuestion: {row['question']}")
    print(f"Ground truth: Should CLARIFY")
    print(f"Model predicted: {row['predicted_action']}")
    print(f"\nRAW MODEL OUTPUT:")
    print(row['raw_output'])
    print("="*80)

DEBUGGING: Let's see actual model outputs


Question: Who is the current chairman of african union commission?
Ground truth: Should CLARIFY
Model predicted: Answer

RAW MODEL OUTPUT:
Action: Answer
Reasoning: The question is not ambiguous because it specifies the exact position and organization. This combination uniquely identifies one individual who holds the current role as the chairman of the African Union Commission.
Facets: []
Response: Moussa Faki Mahamat

Question: Who won the final hoh big brother 20?
Ground truth: Should CLARIFY
Model predicted: Answer

RAW MODEL OUTPUT:
Action: Answer
Reasoning: The question is not ambiguous because it specifies the exact season (big brother 20) and the type of event (final house hold, or HOH). This provides enough detail to uniquely determine the winner of that specific event.
Facets: []
Response: Kyla

Question: How long do contestants get to answer on jeopardy?
Ground truth: Should CLARIFY
Model predicted: Answer

RAW MODEL OUTPUT:
Action:

In [16]:
# Cell: Also check if parsing is working correctly
print("\nDEBUGGING: Testing parser\n")

# Manually test one ambiguous question
test_ambiguous = "Who won the championship?"
raw = generate_response_correct(test_ambiguous, temperature=0.1)

print(f"Question: {test_ambiguous}")
print(f"\nRaw output:\n{raw}")
print(f"\nParsed:")
parsed = parse_model_output(raw)
for k, v in parsed.items():
    print(f"  {k}: {v}")


DEBUGGING: Testing parser

Question: Who won the championship?

Raw output:
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which sport, event category, or time period is being referred to. Without this information, it's unclear what specific championship is being asked about.
Facets: [Sport Type, Event Category, Time Period]
Response: In which sport, event category, and time period are you referring to when asking who won the championship?

Parsed:
  action: Clarify
  reasoning: The question is ambiguous because it does not specify which sport, event category, or time period is being referred to. Without this information, it's unclear what specific championship is being asked about.
  facets: ['Sport Type', 'Event Category', 'Time Period']
  response: In which sport, event category, and time period are you referring to when asking who won the championship?


In [18]:
# Cell: Check what makes these "ambiguous" in the dataset
print("CHECKING GROUND TRUTH ANNOTATIONS\n")
print("="*80)

# Look at the first few "ambiguous" examples from eval_data
for i in range(min(5, len(eval_data))):
    example = eval_data[i]
    if classify_ambiguity(example):
        print(f"\nQuestion: {example['question']}")
        print(f"Type: {example['annotations']['type']}")
        print(f"Annotations: {example['annotations']}")
        print("="*80)

CHECKING GROUND TRUTH ANNOTATIONS



In [20]:
# Cell: Check distribution in eval set vs training set
print("DISTRIBUTION CHECK\n")

# Eval set
eval_ambiguous_count = sum(1 for ex in eval_data if classify_ambiguity(ex))
print(f"Eval set: {eval_ambiguous_count}/{len(eval_data)} ambiguous ({eval_ambiguous_count/len(eval_data):.1%})")

# Training set - load your CSV
import pandas as pd
train_df = pd.read_csv('final_SFT_data_for_use.csv')
train_ambiguous_count = train_df['is_ambiguous'].sum()
print(f"Train set: {train_ambiguous_count}/{len(train_df)} ambiguous ({train_ambiguous_count/len(train_df):.1%})")

# Check if there's a mismatch
print(f"\nAre the distributions similar? {abs(eval_ambiguous_count/len(eval_data) - train_ambiguous_count/len(train_df)) < 0.1}")

DISTRIBUTION CHECK

Eval set: 1070/2002 ambiguous (53.4%)
Train set: 4747/10036 ambiguous (47.3%)

Are the distributions similar? True


In [21]:
# Cell: Deep dive into a specific "ambiguous" example
print("DEEP DIVE INTO ANNOTATIONS\n")
print("="*80)

# Find the example "Who is the current chairman of african union commission?"
for i, ex in enumerate(eval_data):
    if "chairman of african union" in ex['question'].lower():
        print(f"Question: {ex['question']}")
        print(f"\nFull annotations:")
        import json
        print(json.dumps(ex['annotations'], indent=2))
        
        # Check if there are actually multiple QA pairs
        if 'qaPairs' in ex['annotations']:
            qa_pairs = ex['annotations']['qaPairs']
            print(f"\nNumber of QA pairs: {len(qa_pairs)}")
            for idx, pair in enumerate(qa_pairs):
                print(f"\nQA Pair {idx+1}:")
                print(f"  Question: {pair.get('question', 'N/A')}")
                print(f"  Answer: {pair.get('answer', 'N/A')}")
        break

DEEP DIVE INTO ANNOTATIONS

Question: Who is the current chairman of african union commission?

Full annotations:
{
  "type": [
    "multipleQAs",
    "multipleQAs"
  ],
  "answer": [
    [],
    []
  ],
  "qaPairs": [
    {
      "question": [
        "Who is the 4th chairman of african union commission?",
        "Who is the 3rd chairman of african union commission?",
        "Who is the 2nd chairman of african union commission?"
      ],
      "answer": [
        [
          "Moussa Faki",
          "Moussa Faki Mahamat"
        ],
        [
          "Nkosazana Clarice Dlamini-Zuma",
          "Nkosazana Dlamini-Zuma"
        ],
        [
          "Jean Ping"
        ]
      ]
    },
    {
      "question": [
        "Who became the chairman of african union commission in 2017?",
        "Who became the chairman of african union commission in 2012?",
        "Who became the chairman of african union commission in 2008?"
      ],
      "answer": [
        [
          "Moussa Faki",

In [22]:
# Cell: Check if AmbigNQ "multipleQAs" actually has multiple interpretations
# Let's find an example that's marked as multipleQAs and see what's in it
print("CHECKING ACTUAL multipleQAs EXAMPLES\n")
print("="*80)

count = 0
for ex in eval_data:
    if classify_ambiguity(ex) and count < 3:
        print(f"\nQuestion: {ex['question']}")
        
        annotations = ex['annotations']
        print(f"Type: {annotations['type']}")
        
        if 'qaPairs' in annotations:
            qa_pairs = annotations['qaPairs']
            print(f"Number of QA pairs: {len(qa_pairs)}")
            
            # Show each QA pair
            for idx, pair in enumerate(qa_pairs[:3]):  # Show first 3
                print(f"\n  Interpretation {idx+1}:")
                if 'question' in pair and pair['question']:
                    print(f"    Disambiguated Q: {pair['question']}")
                if 'answer' in pair and pair['answer']:
                    print(f"    Answer: {pair['answer']}")
        
        count += 1
        print("="*80)

CHECKING ACTUAL multipleQAs EXAMPLES


Question: Who is the current chairman of african union commission?
Type: ['multipleQAs', 'multipleQAs']
Number of QA pairs: 2

  Interpretation 1:
    Disambiguated Q: ['Who is the 4th chairman of african union commission?', 'Who is the 3rd chairman of african union commission?', 'Who is the 2nd chairman of african union commission?']
    Answer: [['Moussa Faki', 'Moussa Faki Mahamat'], ['Nkosazana Clarice Dlamini-Zuma', 'Nkosazana Dlamini-Zuma'], ['Jean Ping']]

  Interpretation 2:
    Disambiguated Q: ['Who became the chairman of african union commission in 2017?', 'Who became the chairman of african union commission in 2012?', 'Who became the chairman of african union commission in 2008?']
    Answer: [['Moussa Faki', 'Moussa Faki Mahamat'], ['Nkosazana Clarice Dlamini-Zuma', 'Nkosazana Dlamini-Zuma', 'NDZ'], ['Jean Ping']]

Question: Who won the final hoh big brother 20?
Type: ['multipleQAs']
Number of QA pairs: 1

  Interpretation 1:
    Disa

In [24]:
# Cell: Check training data quality - did you have examples like these?
import pandas as pd
train_df = pd.read_csv('final_SFT_data_for_use.csv')

# Look for similar patterns in training data
print("CHECKING TRAINING DATA FOR SIMILAR EXAMPLES\n")
print("="*80)

# Find examples with "current" or "chairman" 
similar_train = train_df[
    (train_df['question'].str.contains('current', case=False, na=False)) |
    (train_df['question'].str.contains('chairman', case=False, na=False))
].head(5)

for idx, row in similar_train.iterrows():
    print(f"\nQuestion: {row['question']}")
    print(f"Is ambiguous: {row['is_ambiguous']}")
    print(f"Action: {row['action']}")
    print(f"Facets: {row['facets']}")
    print(f"Response: {row['positive_response'][:150]}...")
    print("-"*80)

CHECKING TRAINING DATA FOR SIMILAR EXAMPLES


Question: Current captain of the england mens test cricket team?
Is ambiguous: True
Action: Clarify
Facets: ['Sequence Number']
Response: Are you asking about the current captain of England's men's national cricket team, or perhaps a different sequence or context?...
--------------------------------------------------------------------------------

Question: Who is the current receiver general of canada?
Is ambiguous: True
Action: Clarify
Facets: ['Current Status', 'Position Rank']
Response: Are you asking about the current holder of the Receiver General position in Canada, and whether they are in a high-ranking official role?...
--------------------------------------------------------------------------------

Question: Who is the current archbishop of los angeles?
Is ambiguous: True
Action: Clarify
Facets: ['Sequence Number']
Response: Could you specify if this is referring to the sequence number of the current archbishop or another aspect 

In [25]:
# Cell: Check if your training data has temporal ambiguity examples
print("\nCHECKING FOR TEMPORAL AMBIGUITY IN TRAINING\n")
print("="*80)

# Count different types of facets in training data
import ast

def extract_facets(facet_str):
    try:
        return ast.literal_eval(facet_str) if isinstance(facet_str, str) else facet_str
    except:
        return []

train_df['facets_list'] = train_df['facets'].apply(extract_facets)

# Find temporal-related facets
temporal_examples = train_df[
    train_df['facets_list'].apply(lambda x: any('time' in str(f).lower() or 'year' in str(f).lower() or 'current' in str(f).lower() for f in x if f))
]

print(f"Found {len(temporal_examples)} examples with temporal facets")
print(f"\nSample temporal ambiguity examples:")

for idx, row in temporal_examples.head(3).iterrows():
    print(f"\nQ: {row['question']}")
    print(f"Facets: {row['facets_list']}")
    print(f"Response: {row['positive_response'][:150]}...")
    print("-"*80)


CHECKING FOR TEMPORAL AMBIGUITY IN TRAINING

Found 1556 examples with temporal facets

Sample temporal ambiguity examples:

Q: When did the manhattan project began and end?
Facets: ['Time Frame Interpretation']
Response: In which specific year or time period are you interested in knowing about the Manhattan Project's start and end?...
--------------------------------------------------------------------------------

Q: Who sing play that funky music white boy?
Facets: ['Year']
Response: In which year do you want to know who performed "Play That Funky Music White Boy"?...
--------------------------------------------------------------------------------

Q: Voice of the snake in the jungle book?
Facets: ['Film Year', 'Medium of Adaptation']
Response: Are you asking about the voice actor for the snake in the original 1967 Disney film or a more recent adaptation, such as the 2016 live-action version?...
--------------------------------------------------------------------------------


In [29]:
# Cell: Test with SIMPLIFIED format (no facets, just action + response)
def generate_response_simple(question, max_new_tokens=256, temperature=0.7):
    system_msg = (
        "You are a question understanding assistant. "
        "For each question, decide if it is ambiguous or not.\n\n"
        "If AMBIGUOUS (multiple valid interpretations): Ask a clarifying question.\n"
        "If UNAMBIGUOUS (one clear answer): Give a direct answer.\n\n"
        "Format:\n"
        "Decision: Ambiguous|Unambiguous\n"
        "Response: [your clarifying question OR direct answer]\n"
    )
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": question}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "assistant\n" in response:
        response = response.split("assistant\n")[-1]
    elif "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "")
    
    return response.strip()

# Test on a few examples
test_questions = [
    "Who won the championship?",  # Ambiguous
    "When was the frozen ride open at epcot?",  # Unambiguous
    "Who is the current chairman of african union commission?",  # Ambiguous (temporal)
]

print("TESTING SIMPLER FORMAT (asking model to ignore its training)")
print("="*80)
for q in test_questions:
    response = generate_response_simple(q, temperature=0.1)
    print(f"\nQ: {q}")
    print(f"Response:\n{response}")
    print("-"*80)


TESTING SIMPLER FORMAT (asking model to ignore its training)

Q: Who won the championship?
Response:
Decision: Ambiguous
Response: In which sport and during which year did the championship take place?
--------------------------------------------------------------------------------

Q: When was the frozen ride open at epcot?
Response:
Decision: Ambiguous
Response: In which specific year and month were you referring to when asking about the opening of the Frozen ride at EPCOT?
--------------------------------------------------------------------------------

Q: Who is the current chairman of african union commission?
Response:
Decision: Unambiguous
Response: Mr. Moussa Faki Mahamat
--------------------------------------------------------------------------------


# Attempting to restart SFT

In [3]:
import sys
sys.path.insert(0, '/home/bsanghi2/.local/lib/python3.11/site-packages')

import os
import re
import json
import time
from tqdm import tqdm

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
# ===== Load Qwen 32B efficiently =====

MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"

print("🧠 Loading Qwen2.5-32B-Instruct...")

tokenizer_32b = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model_32b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model_32b.eval()  # important for speed

device = model_32b.device
print(f"✅ Model loaded on: {device}")
print(f"✅ CUDA memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


🧠 Loading Qwen2.5-32B-Instruct...


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

✅ Model loaded on: cuda:0
✅ CUDA memory used: 65.53 GB


In [5]:
print("📚 Loading AmbigNQ dataset (light)...")
ds = load_dataset("sewon/ambig_qa", "light")
train_data = ds["train"]
print(f"✅ Train examples: {len(train_data)}")

📚 Loading AmbigNQ dataset (light)...
✅ Train examples: 10036


# FACET GENREATION CODE

In [9]:
# ============================================
# BASIC HELPERS
# ============================================

def classify_ambiguity(example):
    ann = example.get("annotations", {})
    t = ann.get("type", [])
    if isinstance(t, list) and len(t) > 0:
        return t[0] == "multipleQAs"
    if isinstance(t, str):
        return t == "multipleQAs"
    return False


def get_disambiguated_questions(example):
    ann = example.get("annotations", {})
    qa_pairs = ann.get("qaPairs", [])
    out = []
    for pair in qa_pairs:
        q = pair.get("question", None)
        if q is None:
            continue
        if isinstance(q, list):
            out.extend(q)
        else:
            out.append(q)
    # Deduplicate
    seen, uniq = set(), []
    for q in out:
        if q not in seen:
            uniq.append(q)
            seen.add(q)
    return uniq


def extract_facets(text):
    if not text:
        return []
    m = re.search(r"\[FACET\](.*?)[\r\n]*\[/FACET\]", text, flags=re.DOTALL)
    if not m:
        return []

    raw = m.group(1).strip()
    # Try JSON parse
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed]
    except:
        pass

    # Fallback
    raw = raw.strip("[]")
    items = [p.strip().strip('"').strip("'") for p in raw.split(",")]
    return [i for i in items if i]

In [10]:
FACET_SYSTEM = (
    "You are an expert linguist and QA researcher.\n"
    "Your job is to identify abstract facets of ambiguity across disambiguated questions.\n"
    "A facet is a dimension of variation like 'Year', 'Location', 'Sport Type', 'Information Source', etc.\n"
    "Return ONLY the facet list wrapped in [FACET] ... [/FACET]."
)

FACET_FEWSHOTS = """
Example 1:
Ambiguous Question:
Who won the US Open?

Disambiguated Questions:
1. Who won the US Open Tennis Women's Singles in 2023?
2. Who won the US Open Golf Championship in 2019?

Output:
[FACET]["Sport Type", "Event Category", "Year"][/FACET]

Example 2:
Ambiguous Question:
When did Fast and Furious 6 come out?

Disambiguated Questions:
1. When did Fast and Furious 6 premiere in the UK?
2. When did Fast and Furious 6 release in the US?

Output:
[FACET]["Meaning of 'come out'", "Geographical Region"][/FACET]

Example 3:
Ambiguous Question:
Where does India rank in the world economy?

Disambiguated Questions:
1. According to IMF, where does India rank?
2. According to World Bank, where does India rank?

Output:
[FACET]["Information Source"][/FACET]
"""

def build_facet_user_prompt(question, disambig_qs):
    disambig_str = "\n".join(
        f"{i+1}. {q}" for i, q in enumerate(disambig_qs[:6])
    )
    return (
        FACET_FEWSHOTS
        + "\n\nNow do the same for this case.\n\n"
        + f"Ambiguous Question:\n{question}\n\n"
        + "Disambiguated Questions:\n"
        + disambig_str
        + "\n\nOutput ONLY the facet list wrapped in [FACET]...[/FACET]."
    )


In [11]:
# ============================================
# GENERATE FACETS FAST
# ============================================

@torch.inference_mode()
def generate_facets_32b(question, dis_qs, max_new_tokens=64):

    user_prompt = build_facet_user_prompt(question, dis_qs)

    messages = [
        {"role": "system", "content": FACET_SYSTEM},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer_32b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer_32b(prompt, return_tensors="pt").to(model_32b.device)

    output = model_32b.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer_32b.eos_token_id,
        eos_token_id=tokenizer_32b.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output[0, prompt_len:]
    text = tokenizer_32b.decode(new_tokens, skip_special_tokens=True)

    facets = extract_facets(text)
    return facets, text

In [12]:
# ============================================
# MAIN LOOP — FIRST 5000 ROWS ONLY
# ============================================

import pandas as pd

START = 0
END = 5000
SAVE_EVERY = 100
SAVE_PATH = "ambig_facets_0_4999.parquet"

rows = []
print(f"🚀 Generating FACETS ONLY for rows [{START}, {END})...\n")

for i in tqdm(range(START, END), desc="FacetGen", ncols=80):

    ex = train_data[i]
    q = ex["question"]

    # Skip non-ambiguous
    if not classify_ambiguity(ex):
        rows.append({
            "idx": i,
            "question": q,
            "is_ambiguous": False,
            "disambig_questions": [],
            "facets": [],
            "facets_raw": "",
        })
        continue

    dis_qs = get_disambiguated_questions(ex)

    if not dis_qs:
        rows.append({
            "idx": i,
            "question": q,
            "is_ambiguous": True,
            "disambig_questions": [],
            "facets": [],
            "facets_raw": "",
        })
        continue

    # FACET GENERATION
    try:
        facets, raw = generate_facets_32b(q, dis_qs)
    except Exception as e:
        print(f"❌ Error at idx={i}: {e}")
        facets, raw = [], ""

    rows.append({
        "idx": i,
        "question": q,
        "is_ambiguous": True,
        "disambig_questions": dis_qs,
        "facets": facets,
        "facets_raw": raw,
    })

    # Save progress every 100 rows
    if (i - START + 1) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_parquet(SAVE_PATH, index=False)
        print(f"💾 Saved checkpoint to {SAVE_PATH} (up to {i})")

# Final save
df = pd.DataFrame(rows)
df.to_parquet(SAVE_PATH, index=False)
print(f"\n✅ COMPLETED! Saved to {SAVE_PATH}")


🚀 Generating FACETS ONLY for rows [0, 5000)...



FacetGen:   0%|                                        | 0/5000 [00:00<?, ?it/s]/home/bsanghi2/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/bsanghi2/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/bsanghi2/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  war

💾 Saved checkpoint to ambig_facets_0_4999.parquet (up to 199)


FacetGen:   6%|█▊                            | 300/5000 [02:33<29:20,  2.67it/s]

💾 Saved checkpoint to ambig_facets_0_4999.parquet (up to 299)


...

💾 Saved checkpoint to ambig_facets_0_4999.parquet (up to 4599)


FacetGen:  94%|███████████████████████████▎ | 4700/5000 [37:09<02:31,  1.99it/s]

💾 Saved checkpoint to ambig_facets_0_4999.parquet (up to 4699)


FacetGen: 100%|█████████████████████████████| 5000/5000 [39:30<00:00,  2.11it/s]


✅ COMPLETED! Saved to ambig_facets_0_4999.parquet


# I have generated facets paralelly so the code below merges the results

In [13]:
import pandas as pd

# Load both chunks
df1 = pd.read_parquet("ambig_facets_0_4999.parquet")
df2 = pd.read_parquet("ambig_facets_5000_10036.parquet")

print("Part 1 size:", len(df1))
print("Part 2 size:", len(df2))

# Combine
df_full = pd.concat([df1, df2], axis=0).reset_index(drop=True)

print("Merged size:", len(df_full))

# OPTIONAL: Sanity check for duplicates
duplicate_rows = df_full.duplicated(subset=["question"]).sum()
print("Duplicate questions:", duplicate_rows)

# Save merged dataset
df_full.to_parquet("facets_32b_full.parquet", index=False)
df_full.to_csv("facets_32b_full.csv", index=False)

print("✓ Facet dataset merged and saved!")


Part 1 size: 5000
Part 2 size: 5036
Merged size: 10036
Duplicate questions: 0
✓ Facet dataset merged and saved!


In [19]:
df_full

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw
0,0,When did the simpsons first air on television?,True,[When did the Simpsons first air on television...,"[Format Type, Show Type]","[FACET][""Format Type"", ""Show Type""][/FACET]"
1,1,Who played george washington in the john adams...,False,[],[],
2,2,What is the legal age of marriage in usa?,True,"[What is the legal age of marriage, without pa...","[State Specificity, Legal Circumstances]","[FACET][""State Specificity"", ""Legal Circumstan..."
3,3,Who starred in barefoot in the park on broadway?,True,[Who starred in barefoot in the park on broadw...,[Character Role],"[FACET][""Character Role""[/FACET]"
4,4,When did the manhattan project began and end?,True,"[Based on the initial thoughts of the project,...",[Definition of Project Phases],"[FACET][""Definition of Project Phases""[/FACET]"
...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,[When do summer holidays start for schools in ...,[],"[FACET][""Location"", ""Geographical Region""][/FA..."
10032,10032,Who is the band in the movie 10 things i hate ...,True,[What band plays at Club Skunk in the movie 10...,[],"[FACET][""Event Location"", ""Action""][/FACET]"
10033,10033,Who was the last person in the uk to be executed?,False,[],[],
10034,10034,Who does wonder woman end up with in the comics?,True,[Who does wonder woman end up with in All Star...,[],"[FACET][""Publication Series""][/FACET]"


In [33]:
import re

def extract_facet(text):
    """Extract content between [FACET] and [/FACET]"""
    if pd.isna(text) or text == '':
        return []
    
    match = re.search(r'\[FACET\](.*?)\[/FACET\]', str(text), re.DOTALL | re.IGNORECASE)
    if match:
        raw = match.group(1).strip()
        
        # Try to parse as JSON list
        try:
            import json
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                return parsed
        except:
            pass
        
        # Fallback: remove brackets and split
        raw = raw.strip("[]")
        return [f.strip().strip('"').strip("'") for f in raw.split(",") if f.strip()]
    
    return []

# Apply to your dataframe
df_full['extracted_facets'] = df_full['facets_raw'].apply(extract_facet)

# Check results
#print(df[['facets_raw', 'extracted_facets']].head())

In [34]:
df_full

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets
0,0,When did the simpsons first air on television?,True,[When did the Simpsons first air on television...,"[Format Type, Show Type]","[FACET][""Format Type"", ""Show Type""][/FACET]","[Format Type, Show Type]"
1,1,Who played george washington in the john adams...,False,[],[],,[]
2,2,What is the legal age of marriage in usa?,True,"[What is the legal age of marriage, without pa...","[State Specificity, Legal Circumstances]","[FACET][""State Specificity"", ""Legal Circumstan...","[State Specificity, Legal Circumstances]"
3,3,Who starred in barefoot in the park on broadway?,True,[Who starred in barefoot in the park on broadw...,[Character Role],"[FACET][""Character Role""[/FACET]",[Character Role]
4,4,When did the manhattan project began and end?,True,"[Based on the initial thoughts of the project,...",[Definition of Project Phases],"[FACET][""Definition of Project Phases""[/FACET]",[Definition of Project Phases]
...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,[When do summer holidays start for schools in ...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","[Location, Geographical Region]"
10032,10032,Who is the band in the movie 10 things i hate ...,True,[What band plays at Club Skunk in the movie 10...,[],"[FACET][""Event Location"", ""Action""][/FACET]","[Event Location, Action]"
10033,10033,Who was the last person in the uk to be executed?,False,[],[],,[]
10034,10034,Who does wonder woman end up with in the comics?,True,[Who does wonder woman end up with in All Star...,[],"[FACET][""Publication Series""][/FACET]",[Publication Series]


In [35]:
df_full.to_csv('regenerated_sftdata_onlyfacets.csv')

# Regenerating clarifying questions

In [45]:
df_full_first5k=df_full.loc[:5000]
df_full_first5k

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets
0,0,When did the simpsons first air on television?,True,[When did the Simpsons first air on television...,"[Format Type, Show Type]","[FACET][""Format Type"", ""Show Type""][/FACET]","[Format Type, Show Type]"
1,1,Who played george washington in the john adams...,False,[],[],,[]
2,2,What is the legal age of marriage in usa?,True,"[What is the legal age of marriage, without pa...","[State Specificity, Legal Circumstances]","[FACET][""State Specificity"", ""Legal Circumstan...","[State Specificity, Legal Circumstances]"
3,3,Who starred in barefoot in the park on broadway?,True,[Who starred in barefoot in the park on broadw...,[Character Role],"[FACET][""Character Role""[/FACET]",[Character Role]
4,4,When did the manhattan project began and end?,True,"[Based on the initial thoughts of the project,...",[Definition of Project Phases],"[FACET][""Definition of Project Phases""[/FACET]",[Definition of Project Phases]
...,...,...,...,...,...,...,...
4996,4996,When did france lose the world cup final?,True,[When did France lose in the FIFA World Cup fi...,[Sport Type],"[FACET][""Sport Type""][/FACET]",[Sport Type]
4997,4997,Who played galen in planet of the apes?,True,[Who played galen in the 1968 film planet of t...,[Medium],"[FACET][""Medium""][/FACET]",[Medium]
4998,4998,Who wrote like a fox on the run?,True,"[Who wrote like a fox on the run in 1974?, Who...",[Year],"[FACET][""Year""][/FACET]",[Year]
4999,4999,When was kolhapur state merged into bombay pro...,False,[],[],,[]


In [48]:
import torch
import json
from tqdm import tqdm

In [58]:
CQ_FEWSHOTS = """
### EXAMPLES ###

Example 1:
Question: "Who won the US Open?"
Facets: ["Sport Type", "Event Category", "Year"]
Clarifying Question: "Which sport, category, and year of the US Open are you referring to?"

Example 2:
Question: "When did Fast and Furious 6 come out?"
Facets: ["Meaning of 'come out'", "Region"]
Clarifying Question: "Do you mean the theatrical release, digital release, or premiere, and in which region?"

Example 3:
Question: "Where does India rank in the world economy?"
Facets: ["Data Source", "Year"]
Clarifying Question: "Which data source and which year’s ranking are you asking about?"

Example 4:
Question: "Who played Batman?"
Facets: ["Movie Version", "Actor Era"]
Clarifying Question: "Which Batman version or era are you referring to?"
"""

In [59]:
CQ_SYSTEM = f"""
You are an expert ambiguity-resolution agent.
Given an ambiguous question and a list of facets, generate ONE clarifying question.

Rules:
- Ask ONLY ONE question.
- It must address ALL facets naturally.
- It must be conversational and short.
- Do NOT answer the question.
- Do NOT explain your reasoning.
- Output ONLY the clarifying question.

{CQ_FEWSHOTS}
"""

In [60]:
def generate_clarifying_question(question, facets):
    if not isinstance(facets, list) or len(facets) == 0:
        return None   # Not ambiguous → no clarifying question
    
    facets_json = json.dumps(facets)

    messages = [
        {"role": "system", "content": CQ_SYSTEM},
        {"role": "user", "content": 
            f"Question: {question}\nFacets: {facets_json}\nGenerate the clarifying question:"}
    ]

    # Qwen chat formatting
    prompt = tokenizer_32b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_32b(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model_32b.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer_32b.eos_token_id
        )

    text = tokenizer_32b.decode(output[0], skip_special_tokens=False)

    # extract assistant reply
    if "<|im_start|>assistant" in text:
        ans = text.split("<|im_start|>assistant")[-1]
        ans = ans.split("<|im_end|>")[0].strip()
    else:
        ans = text.strip()

    # Make sure it ends with ?
    if not ans.endswith("?"):
        ans = ans.rstrip(".") + "?"

    return ans

In [65]:
def run_clarifying_generation(df):
    df = df.copy()
    df["clarifying_question"] = None

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        if row["is_ambiguous"] and isinstance(row["extracted_facets"], list) and len(row["extracted_facets"]) > 0:
            df.loc[idx, "clarifying_question"] = generate_clarifying_question(
                row["question"], row["extracted_facets"]
            )
        else:
            df.loc[idx, "clarifying_question"] = None  # not ambiguous → no CQ

    return df

In [66]:
print(generate_clarifying_question(
    "Who won the US Open?",
    ["Sport Type", "Event Category", "Year"]
))

Which sport, event category, and year of the US Open are you asking about?


In [67]:
df1 = df_full_first5k
df1 = run_clarifying_generation(df1)
df1.to_csv("clarifying_batch1.csv", index=False)

100%|██████████| 5001/5001 [51:38<00:00,  1.61it/s]  


In [68]:
df_full_last5k=df_full.loc[5001:]
df_full_last5k

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets
5001,5001,What role does finn wolfhard play in it?,True,[What role does Finn Wolfhard play in the 2017...,[],"[FACET][""Year"", ""Movie Part""][/FACET]","[Year, Movie Part]"
5002,5002,Who scored the most goals in football career?,True,[Who has scored the most goals in their footba...,[],"[FACET][""Career Status"", ""Gender"", ""Type of Go...","[Career Status, Gender, Type of Goals]"
5003,5003,When did the song the fighter come out?,True,"[When did the song ""The Fighter"" by Keith Urba...",[],"[FACET][""Artist"", ""Song Title""][/FACET]","[Artist, Song Title]"
5004,5004,When did the guy who played robbie rotten die?,False,[],[],,[]
5005,5005,Whats the latest episode of the walking dead?,True,[What episode number is the latest episode of ...,[],"[FACET][""Type of Information Requested""][/FACET]",[Type of Information Requested]
...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,[When do summer holidays start for schools in ...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","[Location, Geographical Region]"
10032,10032,Who is the band in the movie 10 things i hate ...,True,[What band plays at Club Skunk in the movie 10...,[],"[FACET][""Event Location"", ""Action""][/FACET]","[Event Location, Action]"
10033,10033,Who was the last person in the uk to be executed?,False,[],[],,[]
10034,10034,Who does wonder woman end up with in the comics?,True,[Who does wonder woman end up with in All Star...,[],"[FACET][""Publication Series""][/FACET]",[Publication Series]


In [70]:
df2 = df_full_last5k
df2 = run_clarifying_generation(df2)
df2.to_csv("clarifying_batch2.csv", index=False)

100%|██████████| 5035/5035 [49:05<00:00,  1.71it/s]  


In [71]:
df2

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question
5001,5001,What role does finn wolfhard play in it?,True,[What role does Finn Wolfhard play in the 2017...,[],"[FACET][""Year"", ""Movie Part""][/FACET]","[Year, Movie Part]","Which part and year of ""It"" are you referring to?"
5002,5002,Who scored the most goals in football career?,True,[Who has scored the most goals in their footba...,[],"[FACET][""Career Status"", ""Gender"", ""Type of Go...","[Career Status, Gender, Type of Goals]","""Are you asking about an active or retired pla..."
5003,5003,When did the song the fighter come out?,True,"[When did the song ""The Fighter"" by Keith Urba...",[],"[FACET][""Artist"", ""Song Title""][/FACET]","[Artist, Song Title]","""Could you specify which artist and exact song..."
5004,5004,When did the guy who played robbie rotten die?,False,[],[],,[],None
5005,5005,Whats the latest episode of the walking dead?,True,[What episode number is the latest episode of ...,[],"[FACET][""Type of Information Requested""][/FACET]",[Type of Information Requested],"""Are you looking for the episode number, title..."
...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,[When do summer holidays start for schools in ...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","[Location, Geographical Region]","""In which location or geographical region are ..."
10032,10032,Who is the band in the movie 10 things i hate ...,True,[What band plays at Club Skunk in the movie 10...,[],"[FACET][""Event Location"", ""Action""][/FACET]","[Event Location, Action]",Are you asking about the band playing at a spe...
10033,10033,Who was the last person in the uk to be executed?,False,[],[],,[],None
10034,10034,Who does wonder woman end up with in the comics?,True,[Who does wonder woman end up with in All Star...,[],"[FACET][""Publication Series""][/FACET]",[Publication Series],Which publication series of Wonder Woman are y...


In [72]:
clarify_df=pd.concat([df1, df2], axis=0, ignore_index=True)
clarify_df.shape

(10036, 8)

In [73]:
clarify_df

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question
0,0,When did the simpsons first air on television?,True,[When did the Simpsons first air on television...,"[Format Type, Show Type]","[FACET][""Format Type"", ""Show Type""][/FACET]","[Format Type, Show Type]","""Are you asking about the first air date of Th..."
1,1,Who played george washington in the john adams...,False,[],[],,[],None
2,2,What is the legal age of marriage in usa?,True,"[What is the legal age of marriage, without pa...","[State Specificity, Legal Circumstances]","[FACET][""State Specificity"", ""Legal Circumstan...","[State Specificity, Legal Circumstances]",Are you asking about a specific state and unde...
3,3,Who starred in barefoot in the park on broadway?,True,[Who starred in barefoot in the park on broadw...,[Character Role],"[FACET][""Character Role""[/FACET]",[Character Role],"""Are you asking about the actors who played sp..."
4,4,When did the manhattan project began and end?,True,"[Based on the initial thoughts of the project,...",[Definition of Project Phases],"[FACET][""Definition of Project Phases""[/FACET]",[Definition of Project Phases],"""Are you asking about the start and end dates ..."
...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,[When do summer holidays start for schools in ...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","[Location, Geographical Region]","""In which location or geographical region are ..."
10032,10032,Who is the band in the movie 10 things i hate ...,True,[What band plays at Club Skunk in the movie 10...,[],"[FACET][""Event Location"", ""Action""][/FACET]","[Event Location, Action]",Are you asking about the band playing at a spe...
10033,10033,Who was the last person in the uk to be executed?,False,[],[],,[],None
10034,10034,Who does wonder woman end up with in the comics?,True,[Who does wonder woman end up with in All Star...,[],"[FACET][""Publication Series""][/FACET]",[Publication Series],Which publication series of Wonder Woman are y...


In [75]:
from datasets import load_dataset

ds = load_dataset("sewon/ambig_qa", "light")
data = ds['train']
print(len(data))

10036


In [76]:
data[1]

{'id': '4790842463458965203',
 'question': 'Who played george washington in the john adams series?',
 'annotations': {'type': ['singleAnswer'],
  'answer': [['David Morse']],
  'qaPairs': [{'question': [], 'answer': []}]}}

# Reason generating code

In [77]:
REASONING_PROMPT = """You are an expert in question understanding and ambiguity analysis.
Your job is to produce a short scientific explanation (2–4 lines) describing whether the question is ambiguous and why.

Follow this exact style:

Example 1 (Ambiguous):
Question: Who won the US Open?
Facets: ["Sport Type", "Event Category", "Year"]
Reasoning: The question is ambiguous because "US Open" refers to multiple sports, and the event category and year are unspecified. These missing facets prevent identifying a single answer.

Example 2 (Not Ambiguous):
Question: When did Fast and Furious 6 release in the US?
Facets: []
Reasoning: The question is not ambiguous because it specifies the movie title and the region, providing enough context for a single answer.

Now generate reasoning for the following:

Question: {question}
Facets: {facets}

Reasoning:"""

In [78]:
def generate_reasoning_short(question, facets_list):
    facets_str = str(facets_list)

    prompt = REASONING_PROMPT.format(
        question=question,
        facets=facets_str
    )

    inputs = tokenizer_32b(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_32b.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.0,      # 🔥 deterministic output
            do_sample=False,
            pad_token_id=tokenizer_32b.eos_token_id
        )

    # decode only the generated continuation
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, prompt_len:]
    text = tokenizer_32b.decode(new_tokens, skip_special_tokens=True)

    return text.strip()

In [81]:
import ast
from tqdm import tqdm
import pandas as pd

def generate_reasoning_batch(df, save_path):
    subset = df
    reasoning_list = []

    print(f"🚀 Generating reasoning for rows {start}–{end-1}...")
    for i, row in tqdm(subset.iterrows(), total=len(subset)):
        q = row["question"]
        
        # facets parsed into list
        f = row["facets"]
        if isinstance(f, str):
            try:
                f = ast.literal_eval(f)
            except:
                f = []

        reasoning = generate_reasoning_short(q, f)
        reasoning_list.append(reasoning)

    subset["reasoning"] = reasoning_list

    subset.to_csv(save_path, index=False)
    print(f"✅ Saved: {save_path}")
    return subset

In [82]:
reasoning_0_4999 = generate_reasoning_batch(
    df=df1,
    save_path="reasoning_0_4999.csv"
)

🚀 Generating reasoning for rows 1764466130.905329–1764466130.076111...


  0%|          | 0/5001 [00:00<?, ?it/s]/home/bsanghi2/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/bsanghi2/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/bsanghi2/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
100%|██████████| 5001/5001 [

✅ Saved: reasoning_0_4999.csv


In [1]:
import pandas as pd
import numpy as np
df1=pd.read_csv('reasoning_0_4999.csv')
df1

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question,reasoning
0,0,When did the simpsons first air on television?,True,['When did the Simpsons first air on televisio...,['Format Type' 'Show Type'],"[FACET][""Format Type"", ""Show Type""][/FACET]","['Format Type', 'Show Type']","""Are you asking about the first air date of Th...",The question is ambiguous because it does not ...
1,1,Who played george washington in the john adams...,False,[],[],NaN,[],NaN,The question is not ambiguous because it speci...
2,2,What is the legal age of marriage in usa?,True,"['What is the legal age of marriage, without p...",['State Specificity' 'Legal Circumstances'],"[FACET][""State Specificity"", ""Legal Circumstan...","['State Specificity', 'Legal Circumstances']",Are you asking about a specific state and unde...,"The question is ambiguous because the ""legal a..."
3,3,Who starred in barefoot in the park on broadway?,True,['Who starred in barefoot in the park on broad...,['Character Role'],"[FACET][""Character Role""[/FACET]",['Character Role'],"""Are you asking about the actors who played sp...",The question is ambiguous because while it spe...
4,4,When did the manhattan project began and end?,True,['Based on the initial thoughts of the project...,['Definition of Project Phases'],"[FACET][""Definition of Project Phases""[/FACET]",['Definition of Project Phases'],"""Are you asking about the start and end dates ...",The question is ambiguous because it does not ...
...,...,...,...,...,...,...,...,...,...
4996,4996,When did france lose the world cup final?,True,['When did France lose in the FIFA World Cup f...,['Sport Type'],"[FACET][""Sport Type""][/FACET]",['Sport Type'],"""Are you asking about a specific sport where F...",The question is ambiguous because while it spe...
4997,4997,Who played galen in planet of the apes?,True,['Who played galen in the 1968 film planet of ...,['Medium'],"[FACET][""Medium""][/FACET]",['Medium'],"""Are you asking about the TV series or the mov...","The question is ambiguous because ""Planet of t..."
4998,4998,Who wrote like a fox on the run?,True,['Who wrote like a fox on the run in 1974?'\n ...,['Year'],"[FACET][""Year""][/FACET]",['Year'],"""In which year was 'Like a Fox on the Run' wri...","The question is ambiguous because ""like a fox ..."
4999,4999,When was kolhapur state merged into bombay pro...,False,[],[],NaN,[],NaN,The question is not ambiguous because it speci...


In [6]:
# Cell: Check if clarifying questions explain the ambiguity well
print("🎯 CHECKING: Do clarifying questions explain WHY it's ambiguous?\n")
print("="*80)

# Look at "current" examples
current_samples = df1[df1['question'].str.contains('current', case=False, na=False) & 
                     (df1['is_ambiguous'] == True)].head(5)

for idx, row in current_samples.iterrows():
    print(f"\nQ: {row['question']}")
    print(f"Facets: {row['extracted_facets']}")
    print(f"Clarifying Q: {row['clarifying_question']}")
    
    # Check if it explains temporal ambiguity
    temporal_words = ['year', 'time', 'period', 'when', 'date', 'current', 'now', 'recent']
    has_temporal_explanation = any(word in str(row['clarifying_question']).lower() for word in temporal_words)
    
    print(f"✓ Explains temporal ambiguity: {'YES ✅' if has_temporal_explanation else 'NO ❌'}")
    print("-" * 80)

🎯 CHECKING: Do clarifying questions explain WHY it's ambiguous?


Q: Current captain of the england mens test cricket team?
Facets: ['Captaincy Term']
Clarifying Q: "Are you asking about the current captain as of a specific date or the most recent captain?"?
✓ Explains temporal ambiguity: YES ✅
--------------------------------------------------------------------------------

Q: Who is the current receiver general of canada?
Facets: ['Position Definition', 'Person by Term']
Clarifying Q: "Are you asking about the specific person holding the position or the role definition of the Receiver General of Canada?"?
✓ Explains temporal ambiguity: NO ❌
--------------------------------------------------------------------------------

Q: Who is the current archbishop of los angeles?
Facets: ['Ordinal Position', 'Time Period']
Clarifying Q: Are you asking about the most recent appointment or during a specific time period?
✓ Explains temporal ambiguity: YES ✅
----------------------------------------

In [10]:
# Cell 2: Check for BAD facets
def parse_facets(facet_str):
    """Parse facets from string"""
    if pd.isna(facet_str) or facet_str == '[]' or facet_str == '':
        return []
    try:
        return json.loads(facet_str) if isinstance(facet_str, str) else facet_str
    except:
        try:
            import ast
            return ast.literal_eval(facet_str)
        except:
            return []

df1['facets_parsed'] = df1['extracted_facets'].apply(parse_facets)

# Check for BAD facets (like "Sequence Number" from before)
print("🔍 FACET QUALITY CHECK:\n")

# Common BAD facet patterns to watch for
bad_patterns = ['sequence', 'number', 'rank order', 'position number']

ambig_df = df1[df1['is_ambiguous'] == True]

print(f"Checking {len(ambig_df)} ambiguous examples...\n")

# Find suspicious facets
suspicious = []
for idx, row in ambig_df.iterrows():
    facets = row['facets_parsed']
    for facet in facets:
        facet_lower = str(facet).lower()
        if any(pattern in facet_lower for pattern in bad_patterns):
            suspicious.append({
                'question': row['question'],
                'extracted_facets': facets,
                'clarifying_question': row['clarifying_question']
            })

if suspicious:
    print(f"⚠️ Found {len(suspicious)} potentially bad examples:")
    for ex in suspicious[:10]:
        print(f"\nQ: {ex['question']}")
        print(f"Facets: {ex['extracted_facets']}")
        print(f"Response: {ex['clarifying_question'][:150]}...")
        print("-" * 80)
else:
    print("✅ No obviously bad facet patterns detected!")

# Check facet statistics
ambig_df['num_facets'] = ambig_df['facets_parsed'].apply(len)

print(f"\n📊 FACET STATISTICS:")
print(f"  Average facets per question: {ambig_df['num_facets'].mean():.2f}")
print(f"  Median: {ambig_df['num_facets'].median():.0f}")
print(f"  Questions with 0 facets: {(ambig_df['num_facets'] == 0).sum()}")
print(f"\n  Distribution:")
print(ambig_df['num_facets'].value_counts().sort_index().head(10))

🔍 FACET QUALITY CHECK:

Checking 2357 ambiguous examples...

⚠️ Found 92 potentially bad examples:

Q: When was the last time eagles were in the superbowl?
Facets: ['Occurrence Number']
Response: "Are you asking about the last specific occurrence or their most recent appearance in the Super Bowl?"?...
--------------------------------------------------------------------------------

Q: When is the next batman telltale coming out?
Facets: ['Type of Release', 'Episode Number', 'Platform']
Response: "Are you asking about the release type, episode number, and platform for the next Batman Telltale game?"?...
--------------------------------------------------------------------------------

Q: When do new episodes of berserk come out?
Facets: ['Episode Number', 'Series Year']
Response: "Are you asking about a specific episode number or the release time for a particular year of the series?"?...
--------------------------------------------------------------------------------

Q: Who sings in the

/tmp/ipykernel_7184/2912002475.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambig_df['num_facets'] = ambig_df['facets_parsed'].apply(len)


# redoing everything was a good decision, our dataset looks much better now! 
# we will merge the two halves of the dataset and prepare for SFT

In [2]:
df2=pd.read_csv('reasoning_5000_10036.csv')
df2

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question,reasoning
0,5001,What role does finn wolfhard play in it?,True,['What role does Finn Wolfhard play in the 201...,[],"[FACET][""Year"", ""Movie Part""][/FACET]","['Year', 'Movie Part']","Which part and year of ""It"" are you referring to?","The question is ambiguous because ""It"" could r..."
1,5002,Who scored the most goals in football career?,True,['Who has scored the most goals in their footb...,[],"[FACET][""Career Status"", ""Gender"", ""Type of Go...","['Career Status', 'Gender', 'Type of Goals']","""Are you asking about an active or retired pla...",The question is ambiguous because it does not ...
2,5003,When did the song the fighter come out?,True,"['When did the song ""The Fighter"" by Keith Urb...",[],"[FACET][""Artist"", ""Song Title""][/FACET]","['Artist', 'Song Title']","""Could you specify which artist and exact song...",The question is ambiguous because it does not ...
3,5004,When did the guy who played robbie rotten die?,False,[],[],NaN,[],NaN,The question is ambiguous because it does not ...
4,5005,Whats the latest episode of the walking dead?,True,['What episode number is the latest episode of...,[],"[FACET][""Type of Information Requested""][/FACET]",['Type of Information Requested'],"""Are you looking for the episode number, title...",The question is ambiguous because it does not ...
...,...,...,...,...,...,...,...,...,...
5030,10031,When do the summer holidays start for schools?,True,['When do summer holidays start for schools in...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","['Location', 'Geographical Region']","""In which location or geographical region are ...",The question is ambiguous because it does not ...
5031,10032,Who is the band in the movie 10 things i hate ...,True,['What band plays at Club Skunk in the movie 1...,[],"[FACET][""Event Location"", ""Action""][/FACET]","['Event Location', 'Action']",Are you asking about the band playing at a spe...,The question is not ambiguous because it speci...
5032,10033,Who was the last person in the uk to be executed?,False,[],[],NaN,[],NaN,The question is ambiguous because it does not ...
5033,10034,Who does wonder woman end up with in the comics?,True,['Who does wonder woman end up with in All Sta...,[],"[FACET][""Publication Series""][/FACET]",['Publication Series'],Which publication series of Wonder Woman are y...,The question is not ambiguous because it speci...


In [3]:
reason_df=pd.concat([df1, df2], axis=0, ignore_index=True)
reason_df.shape

(10036, 9)

In [4]:
reason_df

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question,reasoning
0,0,When did the simpsons first air on television?,True,['When did the Simpsons first air on televisio...,['Format Type' 'Show Type'],"[FACET][""Format Type"", ""Show Type""][/FACET]","['Format Type', 'Show Type']","""Are you asking about the first air date of Th...",The question is ambiguous because it does not ...
1,1,Who played george washington in the john adams...,False,[],[],NaN,[],NaN,The question is not ambiguous because it speci...
2,2,What is the legal age of marriage in usa?,True,"['What is the legal age of marriage, without p...",['State Specificity' 'Legal Circumstances'],"[FACET][""State Specificity"", ""Legal Circumstan...","['State Specificity', 'Legal Circumstances']",Are you asking about a specific state and unde...,"The question is ambiguous because the ""legal a..."
3,3,Who starred in barefoot in the park on broadway?,True,['Who starred in barefoot in the park on broad...,['Character Role'],"[FACET][""Character Role""[/FACET]",['Character Role'],"""Are you asking about the actors who played sp...",The question is ambiguous because while it spe...
4,4,When did the manhattan project began and end?,True,['Based on the initial thoughts of the project...,['Definition of Project Phases'],"[FACET][""Definition of Project Phases""[/FACET]",['Definition of Project Phases'],"""Are you asking about the start and end dates ...",The question is ambiguous because it does not ...
...,...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,['When do summer holidays start for schools in...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","['Location', 'Geographical Region']","""In which location or geographical region are ...",The question is ambiguous because it does not ...
10032,10032,Who is the band in the movie 10 things i hate ...,True,['What band plays at Club Skunk in the movie 1...,[],"[FACET][""Event Location"", ""Action""][/FACET]","['Event Location', 'Action']",Are you asking about the band playing at a spe...,The question is not ambiguous because it speci...
10033,10033,Who was the last person in the uk to be executed?,False,[],[],NaN,[],NaN,The question is ambiguous because it does not ...
10034,10034,Who does wonder woman end up with in the comics?,True,['Who does wonder woman end up with in All Sta...,[],"[FACET][""Publication Series""][/FACET]",['Publication Series'],Which publication series of Wonder Woman are y...,The question is not ambiguous because it speci...


# merging answer for disambiguous question from the original dataset

In [5]:
from datasets import load_dataset

ds = load_dataset("sewon/ambig_qa", "light")

In [6]:
data = ds['train']
print(len(data))

10036


In [7]:
data[10033]

{'id': '9101518012234561119',
 'question': 'Who was the last person in the uk to be executed?',
 'annotations': {'type': ['singleAnswer'],
  'answer': [['Gwynne Evans and Peter Allen']],
  'qaPairs': [{'question': [], 'answer': []}]}}

In [8]:
# Cell: Add direct answers for unambiguous questions
import pandas as pd
from datasets import load_dataset

# Load your current dataframe
df = reason_df

# Load original AmbigNQ
ds = load_dataset("sewon/ambig_qa", "light")
train_data = ds['train']

print(f"Current dataframe: {len(df)} rows")
print(f"AmbigNQ train: {len(train_data)} examples\n")

# Create mapping: question → answer
question_to_answer = {}

for example in train_data:
    question = example['question']
    
    # Check if it's a singleAnswer type
    if 'annotations' in example and example['annotations']:
        annotations = example['annotations']
        
        if 'type' in annotations:
            ann_type = annotations['type'][0] if isinstance(annotations['type'], list) else annotations['type']
            
            # Only for unambiguous (singleAnswer)
            if ann_type == 'singleAnswer':
                if 'answer' in annotations and annotations['answer']:
                    answers = annotations['answer']
                    # Extract first answer, first variant (as string, not list)
                    if answers and answers[0]:
                        answer = answers[0][0] if isinstance(answers[0], list) else answers[0]
                        question_to_answer[question] = answer

print(f"✓ Extracted {len(question_to_answer)} unambiguous answers\n")

# Function to get answer for unambiguous questions
def get_direct_answer(row):
    """Get direct answer for unambiguous questions"""
    # Only update if it's unambiguous
    if row['is_ambiguous'] == False:
        question = row['question']
        if question in question_to_answer:
            return question_to_answer[question]
        else:
            # Keep existing response if available
            return row.get('clarifying_question', '') or row.get('response', '')
    else:
        # For ambiguous, keep the clarifying question
        return row.get('clarifying_question', '') or row.get('response', '')

# Apply to create/update response column
df['response'] = df.apply(get_direct_answer, axis=1)

# Verify
print("📊 VERIFICATION:\n")
print(f"Total rows: {len(df)}")
print(f"Ambiguous: {(df['is_ambiguous'] == True).sum()}")
print(f"Unambiguous: {(df['is_ambiguous'] == False).sum()}")

# Check unambiguous examples
unambig_df = df[df['is_ambiguous'] == False]
print(f"\nUnambiguous with responses: {unambig_df['response'].notna().sum()}")
print(f"Empty responses: {(unambig_df['response'].isna() | (unambig_df['response'] == '')).sum()}")

# Show samples
print("\n✅ SAMPLE UNAMBIGUOUS EXAMPLES:\n")
for idx, row in unambig_df.head(5).iterrows():
    print(f"Q: {row['question']}")
    print(f"A: {row['response']}")
    print("-" * 80)

Current dataframe: 10036 rows
AmbigNQ train: 10036 examples

✓ Extracted 5289 unambiguous answers

📊 VERIFICATION:

Total rows: 10036
Ambiguous: 4747
Unambiguous: 5289

Unambiguous with responses: 5289
Empty responses: 0

✅ SAMPLE UNAMBIGUOUS EXAMPLES:

Q: Who played george washington in the john adams series?
A: David Morse
--------------------------------------------------------------------------------
Q: When did the frozen ride open at epcot?
A: June 21, 2016
--------------------------------------------------------------------------------
Q: Name the landforms that form the boundaries of the peninsular plateau?
A: Aravali Range, Satpura Range, Vindhyan Range
--------------------------------------------------------------------------------
Q: When was the first airplane used in war?
A: Blériot XI
--------------------------------------------------------------------------------
Q: What color is a negative benedict's test?
A: clear blue
--------------------------------------------------

In [9]:
df

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question,reasoning,response
0,0,When did the simpsons first air on television?,True,['When did the Simpsons first air on televisio...,['Format Type' 'Show Type'],"[FACET][""Format Type"", ""Show Type""][/FACET]","['Format Type', 'Show Type']","""Are you asking about the first air date of Th...",The question is ambiguous because it does not ...,"""Are you asking about the first air date of Th..."
1,1,Who played george washington in the john adams...,False,[],[],NaN,[],NaN,The question is not ambiguous because it speci...,David Morse
2,2,What is the legal age of marriage in usa?,True,"['What is the legal age of marriage, without p...",['State Specificity' 'Legal Circumstances'],"[FACET][""State Specificity"", ""Legal Circumstan...","['State Specificity', 'Legal Circumstances']",Are you asking about a specific state and unde...,"The question is ambiguous because the ""legal a...",Are you asking about a specific state and unde...
3,3,Who starred in barefoot in the park on broadway?,True,['Who starred in barefoot in the park on broad...,['Character Role'],"[FACET][""Character Role""[/FACET]",['Character Role'],"""Are you asking about the actors who played sp...",The question is ambiguous because while it spe...,"""Are you asking about the actors who played sp..."
4,4,When did the manhattan project began and end?,True,['Based on the initial thoughts of the project...,['Definition of Project Phases'],"[FACET][""Definition of Project Phases""[/FACET]",['Definition of Project Phases'],"""Are you asking about the start and end dates ...",The question is ambiguous because it does not ...,"""Are you asking about the start and end dates ..."
...,...,...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,['When do summer holidays start for schools in...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","['Location', 'Geographical Region']","""In which location or geographical region are ...",The question is ambiguous because it does not ...,"""In which location or geographical region are ..."
10032,10032,Who is the band in the movie 10 things i hate ...,True,['What band plays at Club Skunk in the movie 1...,[],"[FACET][""Event Location"", ""Action""][/FACET]","['Event Location', 'Action']",Are you asking about the band playing at a spe...,The question is not ambiguous because it speci...,Are you asking about the band playing at a spe...
10033,10033,Who was the last person in the uk to be executed?,False,[],[],NaN,[],NaN,The question is ambiguous because it does not ...,Gwynne Evans and Peter Allen
10034,10034,Who does wonder woman end up with in the comics?,True,['Who does wonder woman end up with in All Sta...,[],"[FACET][""Publication Series""][/FACET]",['Publication Series'],Which publication series of Wonder Woman are y...,The question is not ambiguous because it speci...,Which publication series of Wonder Woman are y...


## removing unecessary quotes

In [10]:
# DIRECT quote removal - no fancy logic
df['response'] = df['response'].astype(str).str.strip()  # Remove whitespace
df['response'] = df['response'].str.strip('"')  # Remove "
df['response'] = df['response'].str.strip("'")  # Remove '
df['response'] = df['response'].str.strip()  # Remove whitespace again

In [11]:
df

,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question,reasoning,response
0,0,When did the simpsons first air on television?,True,['When did the Simpsons first air on televisio...,['Format Type' 'Show Type'],"[FACET][""Format Type"", ""Show Type""][/FACET]","['Format Type', 'Show Type']","""Are you asking about the first air date of Th...",The question is ambiguous because it does not ...,Are you asking about the first air date of The...
1,1,Who played george washington in the john adams...,False,[],[],NaN,[],NaN,The question is not ambiguous because it speci...,David Morse
2,2,What is the legal age of marriage in usa?,True,"['What is the legal age of marriage, without p...",['State Specificity' 'Legal Circumstances'],"[FACET][""State Specificity"", ""Legal Circumstan...","['State Specificity', 'Legal Circumstances']",Are you asking about a specific state and unde...,"The question is ambiguous because the ""legal a...",Are you asking about a specific state and unde...
3,3,Who starred in barefoot in the park on broadway?,True,['Who starred in barefoot in the park on broad...,['Character Role'],"[FACET][""Character Role""[/FACET]",['Character Role'],"""Are you asking about the actors who played sp...",The question is ambiguous because while it spe...,Are you asking about the actors who played spe...
4,4,When did the manhattan project began and end?,True,['Based on the initial thoughts of the project...,['Definition of Project Phases'],"[FACET][""Definition of Project Phases""[/FACET]",['Definition of Project Phases'],"""Are you asking about the start and end dates ...",The question is ambiguous because it does not ...,Are you asking about the start and end dates o...
...,...,...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,['When do summer holidays start for schools in...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","['Location', 'Geographical Region']","""In which location or geographical region are ...",The question is ambiguous because it does not ...,In which location or geographical region are y...
10032,10032,Who is the band in the movie 10 things i hate ...,True,['What band plays at Club Skunk in the movie 1...,[],"[FACET][""Event Location"", ""Action""][/FACET]","['Event Location', 'Action']",Are you asking about the band playing at a spe...,The question is not ambiguous because it speci...,Are you asking about the band playing at a spe...
10033,10033,Who was the last person in the uk to be executed?,False,[],[],NaN,[],NaN,The question is ambiguous because it does not ...,Gwynne Evans and Peter Allen
10034,10034,Who does wonder woman end up with in the comics?,True,['Who does wonder woman end up with in All Sta...,[],"[FACET][""Publication Series""][/FACET]",['Publication Series'],Which publication series of Wonder Woman are y...,The question is not ambiguous because it speci...,Which publication series of Wonder Woman are y...


In [12]:
import json
import ast
import pandas as pd

def clean_facets_to_json(val):
    """
    Convert Python list-like facet strings into clean JSON list strings.
    Handles:
    - ['Year']
    - ["Year"]
    - ['Sport', 'Year']
    - [] 
    - NaN
    - already-correct JSON
    """
    if pd.isna(val):
        return "[]"

    # If value is already a Python list
    if isinstance(val, list):
        return json.dumps([str(x).strip() for x in val])

    # If it's a string: try to parse it as Python literal
    if isinstance(val, str):
        s = val.strip()

        # Try JSON parse first
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return json.dumps([str(x).strip() for x in parsed])
        except:
            pass

        # Try Python literal eval
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return json.dumps([str(x).strip() for x in parsed])
        except:
            pass

        # If nothing worked → fall back to single facet string
        return json.dumps([s])

    # fallback
    return "[]"


# Apply to your dataframe
df["facets_clean"] = df["extracted_facets"].apply(clean_facets_to_json)

print(df["facets_clean"].head(10))

0                    ["Format Type", "Show Type"]
1                                              []
2    ["State Specificity", "Legal Circumstances"]
3                              ["Character Role"]
4                ["Definition of Project Phases"]
5                                              []
6                                              []
7                                  ["Sport Type"]
8                                        ["Year"]
9                                              []
Name: facets_clean, dtype: object


In [14]:
df.to_csv('new_regen_sft_dataset.csv')

# Building the final SFT dataset

In [15]:
SYSTEM_PROMPT = (
    "You are a question-understanding agent. "
    "Your job is to detect ambiguity, explain your reasoning, extract facets, "
    "and either ask a clarifying question (if ambiguous) or answer directly (if unambiguous). "
    "Always follow this exact output format:\n\n"
    "Action: Clarify|Answer\n"
    "Reasoning: <reasoning>\n"
    "Facets: <json list>\n"
    "Response: <clarifying question or answer>\n"
)

In [16]:
# ===== Convert is_ambiguous → Action label =====
df["action"] = df["is_ambiguous"].apply(lambda x: "Clarify" if x else "Answer")

In [18]:
# ===== VALIDATION CHECK =====
def validate_row(row):
    ok = True
    if not isinstance(row["question"], str) or len(row["question"]) < 2:
        ok = False
    if row["action"] not in ["Clarify", "Answer"]:
        ok = False
    try:
        json.loads(row["facets_clean"])
    except:
        ok = False
    if not isinstance(row["response"], str):
        ok = False
    return ok

df = df[df.apply(validate_row, axis=1)].reset_index(drop=True)
print(f"✅ Valid rows: {len(df)}")

✅ Valid rows: 10036


In [19]:
# ===== Build final SFT records =====
records = []

for _, row in df.iterrows():

    assistant_output = (
        f"Action: {row['action']}\n"
        f"Reasoning: {row['reasoning']}\n"
        f"Facets: {row['facets_clean']}\n"
        f"Response: {row['response']}"
    )

    record = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": assistant_output}
        ]
    }

    records.append(record)

# ===== Save JSONL =====
with open("sft_data.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("🎉 SFT dataset saved to sft_data.jsonl")

🎉 SFT dataset saved to sft_data.jsonl


# SFT training pipeline

In [2]:
pip install --upgrade "transformers>=4.40.2" "tokenizers>=0.22,<0.24" "trl>=0.8.6" "accelerate>=0.30.0"

  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.0
    Uninstalling transformers-4.46.0:
      Successfully uninstalled transformers-4.46.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import tokenizers, transformers
print(tokenizers.__version__)
print(transformers.__version__)

0.22.1
4.57.3


In [17]:
# SFT TRAINING - FINAL VERIFIED VERSION
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
import torch

print("\n===== STEP 1 — LOAD DATASET =====\n")
dataset = load_dataset("json", data_files="sft_data.jsonl")["train"]
print(f"✓ Loaded {len(dataset)} examples")

print("\n===== STEP 2 — LOAD MODEL & TOKENIZER =====\n")
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
print("✓ Model loaded")

print("\n===== STEP 3 — APPLY CHAT TEMPLATE =====\n")
def build_text(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }
dataset = dataset.map(build_text, remove_columns=["messages"])
print("✓ Dataset formatted")

print("\n===== STEP 4 — LORA CONFIG =====\n")
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,  # ← FIXED: alpha = r
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
)
print("✓ LoRA config ready")

print("\n===== STEP 5 — SFT CONFIG =====\n")
sft_config = SFTConfig(
    output_dir="clarifier_sft_output_v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
    max_length=2048,
    packing=False,
)
print("✓ Training config ready")

print("\n===== STEP 6 — INITIALIZE TRAINER =====\n")
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
print("✓ Trainer initialized")

print("\n===== STEP 7 — START TRAINING =====\n")
print(f"Training on {len(dataset)} examples for 2 epochs")
print(f"Effective batch size: {1 * 8} (per_device=1 × grad_accum=8)")
print(f"Total steps: ~{len(dataset) * 2 // 8}")
print("\nStarting training...\n")

trainer.train()

print("\n🎉 TRAINING COMPLETE!\n")

print("===== STEP 8 — SAVE MODEL =====\n")
trainer.save_model("clarifier_final_model_v2")
tokenizer.save_pretrained("clarifier_final_model_v2")
print("✓ Model saved to: clarifier_final_model_v2\n")


===== STEP 1 — LOAD DATASET =====

✓ Loaded 10036 examples

===== STEP 2 — LOAD MODEL & TOKENIZER =====



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Model loaded

===== STEP 3 — APPLY CHAT TEMPLATE =====

✓ Dataset formatted

===== STEP 4 — LORA CONFIG =====

✓ LoRA config ready

===== STEP 5 — SFT CONFIG =====

✓ Training config ready

===== STEP 6 — INITIALIZE TRAINER =====



Adding EOS to train dataset:   0%|          | 0/10036 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10036 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10036 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


✓ Trainer initialized

===== STEP 7 — START TRAINING =====

Training on 10036 examples for 2 epochs
Effective batch size: 8 (per_device=1 × grad_accum=8)
Total steps: ~2509

Starting training...



ValueError: Trainer tried to instantiate bnb optimizer but `bitsandbytes` is not installed!

In [19]:
print("\n===== STEP 7 — START TRAINING =====\n")
print(f"Training on {len(dataset)} examples for 2 epochs")
print(f"Effective batch size: {1 * 8} (per_device=1 × grad_accum=8)")
print(f"Total steps: ~{len(dataset) * 2 // 8}")
print("\nStarting training...\n")

trainer.train()

print("\n🎉 TRAINING COMPLETE!\n")

print("===== STEP 8 — SAVE MODEL =====\n")
trainer.save_model("clarifier_final_model_v2")
tokenizer.save_pretrained("clarifier_final_model_v2")
print("✓ Model saved to: clarifier_final_model_v2\n")


===== STEP 7 — START TRAINING =====

Training on 10036 examples for 2 epochs
Effective batch size: 8 (per_device=1 × grad_accum=8)
Total steps: ~2509

Starting training...



/home/bsanghi2/.local/lib/python3.11/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss
10,2.892900
20,2.389600
30,1.443500
40,0.724900
50,0.508900
60,0.419400
70,0.364500
80,0.377900
90,0.370400
100,0.338000


/home/bsanghi2/.local/lib/python3.11/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/home/bsanghi2/.local/lib/python3.11/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/home/bsanghi2/.local/lib/python3.11/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/


🎉 TRAINING COMPLETE!

===== STEP 8 — SAVE MODEL =====

✓ Model saved to: clarifier_final_model_v2



In [11]:
# Cell: Check versions
import transformers
import trl
import peft

print(f"transformers: {transformers.__version__}")
print(f"trl: {trl.__version__}")
print(f"peft: {peft.__version__}")

transformers: 4.57.3
trl: 0.25.1
peft: 0.18.0


In [12]:
# Cell: Check SFTTrainer signature
from trl import SFTTrainer
import inspect

sig = inspect.signature(SFTTrainer.__init__)
print("SFTTrainer parameters:")
for param in sig.parameters:
    print(f"  - {param}")

SFTTrainer parameters:
  - self
  - model
  - args
  - data_collator
  - train_dataset
  - eval_dataset
  - processing_class
  - compute_loss_func
  - compute_metrics
  - callbacks
  - optimizers
  - optimizer_cls_and_kwargs
  - preprocess_logits_for_metrics
  - peft_config
  - formatting_func


In [16]:
# Cell: Check if SFTConfig exists in your TRL version
try:
    from trl import SFTConfig
    import inspect
    sig = inspect.signature(SFTConfig.__init__)
    print("✅ SFTConfig exists! Parameters:")
    for param in sig.parameters:
        print(f"  - {param}")
except ImportError:
    print("❌ SFTConfig doesn't exist in this TRL version")
    print("We'll use a different approach")

✅ SFTConfig exists! Parameters:
  - self
  - output_dir
  - overwrite_output_dir
  - do_train
  - do_eval
  - do_predict
  - eval_strategy
  - prediction_loss_only
  - per_device_train_batch_size
  - per_device_eval_batch_size
  - per_gpu_train_batch_size
  - per_gpu_eval_batch_size
  - gradient_accumulation_steps
  - eval_accumulation_steps
  - eval_delay
  - torch_empty_cache_steps
  - learning_rate
  - weight_decay
  - adam_beta1
  - adam_beta2
  - adam_epsilon
  - max_grad_norm
  - num_train_epochs
  - max_steps
  - lr_scheduler_type
  - lr_scheduler_kwargs
  - warmup_ratio
  - warmup_steps
  - log_level
  - log_level_replica
  - log_on_each_node
  - logging_dir
  - logging_strategy
  - logging_first_step
  - logging_steps
  - logging_nan_inf_filter
  - save_strategy
  - save_steps
  - save_total_limit
  - save_safetensors
  - save_on_each_node
  - save_only_model
  - restore_callback_states_from_checkpoint
  - no_cuda
  - use_cpu
  - use_mps_device
  - seed
  - data_seed
  - jit_m

# Evaluation

In [21]:
# UPLOAD MODEL TO HUGGING FACE
from huggingface_hub import HfApi, login
import os

print("="*80)
print("UPLOADING MODEL TO HUGGING FACE")
print("="*80)

# ============================================================================
# STEP 1: LOGIN TO HUGGING FACE
# ============================================================================
print("\n📝 STEP 1: Login to Hugging Face\n")

# You'll need your HF token - get it from https://huggingface.co/settings/tokens
# Option A: Login interactively (will prompt for token)
login()

# Option B: Or set token directly
# HF_TOKEN = "hf_your_token_here"
# login(token=HF_TOKEN)

print("✓ Logged in successfully!\n")

# ============================================================================
# STEP 2: UPLOAD MODEL
# ============================================================================
print("📤 STEP 2: Uploading model...\n")

# Set your model name
HF_USERNAME = "bhavanasanghi9"  # Your HF username
MODEL_NAME = "clarifier-sft-qwen-7b"  # Name for your model
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

# Path to your trained model
LOCAL_MODEL_PATH = "clarifier_final_model_v2"

# Upload
api = HfApi()

print(f"Uploading to: {REPO_ID}")
print(f"From: {LOCAL_MODEL_PATH}\n")

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,  # Don't error if repo already exists
    private=False,  # Set to True if you want private repo
)

print("✓ Repository created/verified")

# Upload all files in the model directory
api.upload_folder(
    folder_path=LOCAL_MODEL_PATH,
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"\n🎉 MODEL UPLOADED SUCCESSFULLY!")
print(f"✓ View at: https://huggingface.co/{REPO_ID}")
print("="*80)

UPLOADING MODEL TO HUGGING FACE

📝 STEP 1: Login to Hugging Face



✓ Logged in successfully!

📤 STEP 2: Uploading model...

Uploading to: bhavanasanghi9/clarifier-sft-qwen-7b
From: clarifier_final_model_v2

✓ Repository created/verified


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 MODEL UPLOADED SUCCESSFULLY!
✓ View at: https://huggingface.co/bhavanasanghi9/clarifier-sft-qwen-7b


In [23]:
from datasets import load_dataset

ds = load_dataset("sewon/ambig_qa", "light")
data = ds['validation']

In [3]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import confusion_matrix

# =========================================================
# 1. Load AmbigNQ → Convert to clean evaluation dataframe
# =========================================================

def dataset_to_df(data):
    rows = []
    for entry in data:
        ann = entry["annotations"]
        ann_type = ann["type"][0] if isinstance(ann["type"], list) else ann["type"]

        # flatten single-answer
        flat_answers = []
        if ann_type == "singleAnswer":
            ans_list = ann.get("answer", [])
            for ans in ans_list:
                if isinstance(ans, list):
                    flat_answers.extend(ans)
                elif isinstance(ans, str):
                    flat_answers.append(ans)

        rows.append({
            "question": entry["question"],
            "type": ann_type,
            "answers": flat_answers
        })

    df = pd.DataFrame(rows)
    df["is_ambiguous"] = df["type"] == "multipleQAs"
    df["action"] = np.where(df["is_ambiguous"], "Clarify", "Answer")
    return df

val_df = dataset_to_df(data)     # <-- your AmbigNQ split
print("Eval dataset size:", len(val_df))


# =========================================================
# 2. Load model (base + LoRA merged)
# =========================================================

base_model = "Qwen/Qwen2.5-7B-Instruct"
lora_model = "bhavanasanghi9/clarifier-sft-qwen-7b"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = PeftModel.from_pretrained(model, lora_model)
model = model.merge_and_unload()
model.eval()

print("✓ Finetuned model loaded.")


# =========================================================
# 3. Inference utilities
# =========================================================

SYSTEM_PROMPT = (
    "You are a question understanding agent trained on AmbigQA-style ambiguity.\n"
    "For each user question:\n"
    "1) Decide if it is ambiguous.\n"
    "2) Explain your reasoning.\n"
    "3) List facets if ambiguous.\n"
    "4) Ask a clarifying question OR give a direct answer.\n\n"
    "Format:\n"
    "Action: Clarify|Answer\n"
    "Reasoning: ...\n"
    "Facets: [...]\n"
    "Response: ...\n"
)

def run_inference(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # remove system/user messages if present
    if "<|im_start|>assistant" in decoded:
        decoded = decoded.split("<|im_start|>assistant")[-1]

    return decoded.strip()


def extract_action(text):
    for line in text.split("\n"):
        if line.lower().startswith("action:"):
            act = line.split(":", 1)[1].strip()
            if act in ["Clarify", "Answer"]:
                return act
    return "N/A"


def extract_response(text):
    for line in text.split("\n"):
        if line.lower().startswith("response:"):
            resp = line.split(":", 1)[1].strip()
            resp = resp.replace("<|im_end|>", "").strip()
            return resp
    return ""


# =========================================================
# 4. Evaluate model
# =========================================================

pred_action = []
true_action = []
pred_answer_text = []
true_answers_list = []

print("Running evaluation...")

for idx, row in tqdm(val_df.iterrows(), total=len(val_df)):
    q = row["question"]
    gold_act = row["action"]
    gold_ans = row["answers"]

    out = run_inference(q)
    pred_act = extract_action(out)
    pred_resp = extract_response(out)

    pred_action.append(pred_act)
    true_action.append(gold_act)
    pred_answer_text.append(pred_resp)
    true_answers_list.append(gold_ans)


# =========================================================
# 5. Scoring
# =========================================================

eval_df = pd.DataFrame({
    "question": val_df["question"],
    "true_action": true_action,
    "pred_action": pred_action,
    "true_answers": true_answers_list,
    "pred_response": pred_answer_text
})

# Action accuracy
action_accuracy = (eval_df["true_action"] == eval_df["pred_action"]).mean()
print("\n===== RESULTS =====")
print("Action Accuracy:", round(action_accuracy, 3))

# Semantic answer accuracy
def semantically_correct(gold_answers, pred_resp):
    pred = pred_resp.lower()
    for gold in gold_answers:
        if gold.lower() in pred:
            return True
    return False

eval_df["answer_correct"] = eval_df.apply(
    lambda row: semantically_correct(row["true_answers"], row["pred_response"])
               if row["true_action"] == "Answer" else None,
    axis=1
)

answer_accuracy = eval_df["answer_correct"].dropna().mean()
print("Semantic Answer Accuracy:", round(answer_accuracy, 3))

# Confusion matrix
cm = confusion_matrix(
    eval_df["true_action"],
    eval_df["pred_action"],
    labels=["Answer", "Clarify"]
)
print("\nConfusion Matrix:\n", cm)

ImportError: cannot import name 'AutoTokenizer' from 'transformers' (unknown location)

# preparing DPO dataset

In [29]:
import pandas as pd 
import numpy as np

In [30]:
sft_df=pd.read_csv('new_regen_sft_dataset.csv')
sft_df

,Unnamed: 0,idx,question,is_ambiguous,disambig_questions,facets,facets_raw,extracted_facets,clarifying_question,reasoning,response,facets_clean
0,0,0,When did the simpsons first air on television?,True,['When did the Simpsons first air on televisio...,['Format Type' 'Show Type'],"[FACET][""Format Type"", ""Show Type""][/FACET]","['Format Type', 'Show Type']","""Are you asking about the first air date of Th...",The question is ambiguous because it does not ...,Are you asking about the first air date of The...,"[""Format Type"", ""Show Type""]"
1,1,1,Who played george washington in the john adams...,False,[],[],NaN,[],NaN,The question is not ambiguous because it speci...,David Morse,[]
2,2,2,What is the legal age of marriage in usa?,True,"['What is the legal age of marriage, without p...",['State Specificity' 'Legal Circumstances'],"[FACET][""State Specificity"", ""Legal Circumstan...","['State Specificity', 'Legal Circumstances']",Are you asking about a specific state and unde...,"The question is ambiguous because the ""legal a...",Are you asking about a specific state and unde...,"[""State Specificity"", ""Legal Circumstances""]"
3,3,3,Who starred in barefoot in the park on broadway?,True,['Who starred in barefoot in the park on broad...,['Character Role'],"[FACET][""Character Role""[/FACET]",['Character Role'],"""Are you asking about the actors who played sp...",The question is ambiguous because while it spe...,Are you asking about the actors who played spe...,"[""Character Role""]"
4,4,4,When did the manhattan project began and end?,True,['Based on the initial thoughts of the project...,['Definition of Project Phases'],"[FACET][""Definition of Project Phases""[/FACET]",['Definition of Project Phases'],"""Are you asking about the start and end dates ...",The question is ambiguous because it does not ...,Are you asking about the start and end dates o...,"[""Definition of Project Phases""]"
...,...,...,...,...,...,...,...,...,...,...,...,...
10031,10031,10031,When do the summer holidays start for schools?,True,['When do summer holidays start for schools in...,[],"[FACET][""Location"", ""Geographical Region""][/FA...","['Location', 'Geographical Region']","""In which location or geographical region are ...",The question is ambiguous because it does not ...,In which location or geographical region are y...,"[""Location"", ""Geographical Region""]"
10032,10032,10032,Who is the band in the movie 10 things i hate ...,True,['What band plays at Club Skunk in the movie 1...,[],"[FACET][""Event Location"", ""Action""][/FACET]","['Event Location', 'Action']",Are you asking about the band playing at a spe...,The question is not ambiguous because it speci...,Are you asking about the band playing at a spe...,"[""Event Location"", ""Action""]"
10033,10033,10033,Who was the last person in the uk to be executed?,False,[],[],NaN,[],NaN,The question is ambiguous because it does not ...,Gwynne Evans and Peter Allen,[]
10034,10034,10034,Who does wonder woman end up with in the comics?,True,['Who does wonder woman end up with in All Sta...,[],"[FACET][""Publication Series""][/FACET]",['Publication Series'],Which publication series of Wonder Woman are y...,The question is not ambiguous because it speci...,Which publication series of Wonder Woman are y...,"[""Publication Series""]"


In [32]:
neg_df=pd.read_csv('neg_synthetic_full.csv')
neg_df

,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,neg_response
0,0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,['When did the Simpsons first air on televisio...,True,"['Type of Show', 'Platform']","The Simpsons first aired on December 17, 1989."
1,1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],Could you tell me which season of the John Ada...
2,2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,"['What is the legal age of marriage, without p...",True,"['State', 'Circumstances for Marriage']",The legal age of marriage in the USA is 18.
3,3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,['Who starred in barefoot in the park on broad...,True,['Character Name'],Walter Matthau and Anne Bancroft starred in Ba...
4,4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,['Based on the initial thoughts of the project...,True,['Time Frame Interpretation'],The Manhattan Project began in 1942 and ended ...
...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,multipleQAs,[{'question': ['When do summer holidays start ...,['When do summer holidays start for schools in...,True,"['Geographical Region', 'Specific Location']",Summer holidays start in June.
10032,10032,Who is the band in the movie 10 things i hate ...,multipleQAs,[{'question': ['What band plays at Club Skunk ...,['What band plays at Club Skunk in the movie 1...,True,"['Location', 'Event Type']",No Doubt.
10033,10033,Who was the last person in the uk to be executed?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],Could you specify which method of execution yo...
10034,10034,Who does wonder woman end up with in the comics?,multipleQAs,[{'question': ['Who does wonder woman end up w...,['Who does wonder woman end up with in All Sta...,True,"['Story Universe', 'Publication Series']",Wonder Woman ends up with Steve Trevor in the ...


In [33]:
sft_df_final=pd.merge(sft_df,neg_df[['question', 'neg_response']], on='question', how='left')
sft_df_final=sft_df_final[['question','is_ambiguous','facets_clean','response','reasoning','neg_response']]

In [22]:
sft_df_final.to_csv('regen_dpo_dataset.csv')

In [36]:
sft_df_final=pd.read_csv('regen_dpo_dataset.csv')
sft_df_final.head()

,Unnamed: 0,question,is_ambiguous,facets_clean,response,reasoning,neg_response
0,0,When did the simpsons first air on television?,True,"[""Format Type"", ""Show Type""]",Are you asking about the first air date of The...,The question is ambiguous because it does not ...,"The Simpsons first aired on December 17, 1989."
1,1,Who played george washington in the john adams...,False,[],David Morse,The question is not ambiguous because it speci...,Could you tell me which season of the John Ada...
2,2,What is the legal age of marriage in usa?,True,"[""State Specificity"", ""Legal Circumstances""]",Are you asking about a specific state and unde...,"The question is ambiguous because the ""legal a...",The legal age of marriage in the USA is 18.
3,3,Who starred in barefoot in the park on broadway?,True,"[""Character Role""]",Are you asking about the actors who played spe...,The question is ambiguous because while it spe...,Walter Matthau and Anne Bancroft starred in Ba...
4,4,When did the manhattan project began and end?,True,"[""Definition of Project Phases""]",Are you asking about the start and end dates o...,The question is ambiguous because it does not ...,The Manhattan Project began in 1942 and ended ...


# Evaluating SFT results correctly

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
from datasets import load_dataset

ds = load_dataset("sewon/ambig_qa", "light")
data = ds['validation']

In [25]:
data[0]

{'id': '-807825952267713091',
 'question': 'Who plays the doctor in dexter season 1?',
 'annotations': {'type': ['singleAnswer'],
  'answer': [['Tony Goldwyn', 'Goldwyn']],
  'qaPairs': [{'question': [], 'answer': []}]}}

In [3]:
def dataset_to_df(data):
  rows = []
  for entry in data:
    ann = entry["annotations"]
    ann_type = ann["type"][0] if isinstance(ann["type"], list) else ann["type"]
    single_ans_list = ann.get("answer", [])
    single_ans_flat = []
    if isinstance(single_ans_list, list):
        for ans in single_ans_list:
            if isinstance(ans, list):
                single_ans_flat.extend(ans)
            elif isinstance(ans, str):
                single_ans_flat.append(ans)

    row = {
        "question": entry["question"],
        "type": ann_type,
        "answers": single_ans_flat if ann_type == "singleAnswer" else []
    }

    rows.append(row)

    df = pd.DataFrame(rows)
    df["is_ambiguous"] = df["type"].apply(lambda t: t == "multipleQAs")
    df['action']=np.where(df['is_ambiguous']==True, 'Clarify', 'Answer')

  return df

val_df = dataset_to_df(data)

In [4]:
val_df.head(5)

,question,type,answers,is_ambiguous,action
0,Who plays the doctor in dexter season 1?,singleAnswer,"[Tony Goldwyn, Goldwyn]",False,Answer
1,How often does spermatogeneis—the production o...,singleAnswer,"[usually continues uninterrupted until death, ...",False,Answer
2,When was the first remote control tv invented?,singleAnswer,"[1950, 1950]",False,Answer
3,Why did the st louis cardinals move to arizona?,singleAnswer,"[mediocrity of the Cardinals,a then-21-year-ol...",False,Answer
4,How many episodes are in season 2 of chesapeak...,singleAnswer,"[10, 10]",False,Answer


In [27]:
val_df.iloc[0]

question        Who plays the doctor in dexter season 1?
type                                        singleAnswer
answers                          [Tony Goldwyn, Goldwyn]
is_ambiguous                                       False
action                                            Answer
Name: 0, dtype: object

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# 1. True base model
base_model_name = "Qwen/Qwen2.5-7B-Instruct"

# 2. LoRA adapters
clarifier_lora = "bhavanasanghi9/clarifier-sft-qwen-7b"
#dpo_lora = "nishap4/adv-nlp-sft-dpo-model"

# 3. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# 4. Load full base model
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# 5. Apply Clarifier SFT LoRA
model = PeftModel.from_pretrained(model, clarifier_lora)

# Optional: put in eval mode
model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version', 'qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [9]:
SYSTEM_PROMPT = """
You are a clarification-seeking question-understanding agent.

Your job:
1. Determine whether the user question is AMBIGUOUS.
2. If ambiguous → identify the facets of ambiguity and ask a clarifying question.
3. If unambiguous → answer directly.
4. ALWAYS output in the EXACT format below (no deviations):

Action: Clarify | Answer
Reasoning: <one short paragraph explaining your reasoning>
Facets: [list, of, facets]   # empty list [] if the question is unambiguous
Response: <clarifying question OR direct answer>

Your output MUST contain all four fields.

--------------------
### FEW-SHOT EXAMPLES
--------------------

Example 1:
User Question:
"Who founded Apple?"

Action: Clarify
Reasoning: The question refers to "Apple" but multiple founders exist (Steve Jobs, Steve Wozniak, Ronald Wayne). Clarification is needed to know which person the user is asking about.
Facets: ["Which founder?", "Steve Jobs vs Steve Wozniak vs Ronald Wayne"]
Response: Do you mean Steve Jobs, Steve Wozniak, or Ronald Wayne?

Example 2:
User Question:
"What is 2+2?"

Action: Answer
Reasoning: The question is clear, numeric, and has only one interpretation. No facets of ambiguity exist.
Facets: []
Response: 4

--------------------

Follow the format STRICTLY for every response.
"""


def run_inference(model, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=None,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=False)

    # Extract assistant section
    if "<|im_start|>assistant" in decoded:
        return decoded.split("<|im_start|>assistant")[-1].strip()

    return decoded.strip()

In [10]:
def extract_action(text):
    """
    Extracts `Clarify` or `Answer` from model output.
    Looks for lines like: "Action: Clarify"
    """
    text = text.strip()

    # Search explicitly in formatted lines
    for line in text.split("\n"):
        if line.lower().startswith("action:"):
            action = line.split(":", 1)[1].strip()
            if action in ["Clarify", "Answer"]:
                return action

    return "N/A"

def extract_response(text):
    text = text.strip()

    for line in text.split("\n"):
        if line.lower().startswith("response:"):
            res = line.split(":", 1)[1].strip()
            if "<|im_end|>" in res:
              res = res.split("<|im_end|>")[0].strip()
            return res

    return "N/A"

In [13]:
# INFERENCE

pred_list = []
actual_list = []
pred_res_list = []
real_res_list = []

N_EVAL=500 #-------> change this if you want to evaluate for 500 or 2000 rows

subset=val_df.iloc[:N_EVAL]

for idx, row in tqdm(subset.iterrows(), total=len(subset)):
  question = row["question"]
  true_label = row["action"]
  answers = row["answers"]

  model_output = run_inference(model, question)
  pred_label = extract_action(model_output)
  response = extract_response(model_output)

  # print(idx)
  # print(question)
  # print(model_output)

  actual_list.append(true_label)
  pred_list.append(pred_label)
  pred_res_list.append(response)
  real_res_list.append(answers)

100%|██████████| 500/500 [57:28<00:00,  6.90s/it] 


In [15]:
# EVALUATION

eval_df = pd.DataFrame({
    "question": subset["question"],
    "true_action": actual_list,
    "pred_action": pred_list,
    "true_answers": real_res_list,
    "pred_response": pred_res_list,
})

res_correct = []
for idx, row in eval_df.iterrows():
  true_answers = row['true_answers']
  pred_response = row['pred_response']
  val = False
  for e in true_answers:
    e = e.lower()
    if e in pred_response.lower(): # keyword-match
      val = True
      break
  res_correct.append(val)
eval_df['res_correct'] = res_correct
eval_df['action_correct'] = eval_df['true_action'] == eval_df['pred_action']

In [16]:
print("Model Accuracy", eval_df['action_correct'].mean())

answer_accuracy = eval_df.loc[eval_df["true_action"] == "Answer", "res_correct"].mean()
print("Answer Accuracy:", answer_accuracy)

Model Accuracy 0.648
Answer Accuracy: 0.21304347826086956


In [17]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(eval_df["true_action"], eval_df["pred_action"]))

              precision    recall  f1-score   support

      Answer       0.70      0.41      0.52       230
     Clarify       0.63      0.85      0.72       270

    accuracy                           0.65       500
   macro avg       0.66      0.63      0.62       500
weighted avg       0.66      0.65      0.63       500



In [18]:
cm = confusion_matrix(eval_df["true_action"], eval_df["pred_action"],
                      labels=["Answer", "Clarify"])

cm_df = pd.DataFrame(
    cm,
    index=["True_Answer", "True_Clarify"],
    columns=["Pred_Answer", "Pred_Clarify"]
)

cm_df

,Pred_Answer,Pred_Clarify
True_Answer,95,135
True_Clarify,41,229


In [19]:
print("Clarify real ratio", (eval_df["true_action"] == "Clarify").mean())
print("Clarify pred ratio", (eval_df["pred_action"] == "Clarify").mean())

Clarify real ratio 0.54
Clarify pred ratio 0.728


# UPDATED PIPELINE FROM HERE ON

              Synthetic Qwen 32B pairs (10k)
                           │
                           ▼
               Normalize into strict format
                           │
                           ▼
         Inject 20–30% real SFT model mistakes
          (misclassification miner results)
                           │
                           ▼
                        DPO training
                           │
                           ▼
            Evaluate + mine new misclassifications
                           │
                           ▼
               Optional: DPO round 2 (small)


# some pressure points - 
### 1. since we used qwen 32B model, there are some inconsistencies in the output that was generated. due to time constraints, we do not have time to re-do this. But this could be future work

# Preparing the right DPO dataset

In [9]:
import json
import pandas as pd
from tqdm import tqdm

# 1. Load your original DPO CSV (change filename if needed)
# It should have: question, reasoning, response, facets_clean, neg_response, is_ambiguous
df = pd.read_csv("regen_dpo_dataset.csv")  # <-- change if your file has a different name

# 2. System prompt (same as for SFT/eval)
SYSTEM_PROMPT = """
You are a clarification-seeking question-understanding agent.

Your job:
1. Determine whether the user question is AMBIGUOUS.
2. If ambiguous → identify the facets of ambiguity and ask a clarifying question.
3. If unambiguous → answer directly.
4. ALWAYS output in the EXACT format below (no deviations):

Action: Clarify | Answer
Reasoning: <one short paragraph explaining your reasoning>
Facets: [list, of, facets]   # empty list [] if the question is unambiguous
Response: <clarifying question OR direct answer>

Your output MUST contain all four fields.

--------------------
### FEW-SHOT EXAMPLES
--------------------

Example 1:
User Question:
"Who founded Apple?"

Action: Clarify
Reasoning: The question refers to "Apple" but multiple founders exist (Steve Jobs, Steve Wozniak, Ronald Wayne). Clarification is needed to know which person the user is asking about.
Facets: ["Which founder?", "Steve Jobs vs Steve Wozniak vs Ronald Wayne"]
Response: Do you mean Steve Jobs, Steve Wozniak, or Ronald Wayne?

Example 2:
User Question:
"What is 2+2?"

Action: Answer
Reasoning: The question is clear, numeric, and has only one interpretation. No facets of ambiguity exist.
Facets: []
Response: 4

--------------------

Follow the format STRICTLY for every response.
""".strip()


def build_structured_output(action, reasoning, facets, response):
    return (
        f"Action: {action}\n"
        f"Reasoning: {reasoning}\n"
        f"Facets: {facets}\n"
        f"Response: {response}"
    )


out_path = "dpo_dataset_clean.jsonl"

with open(out_path, "w") as f:
    for _, row in tqdm(df.iterrows(), total=len(df)):
        question = row["question"]
        reasoning = row["reasoning"]
        response = row["response"]
        facets = row["facets_clean"]
        neg_response = row["neg_response"]
        is_ambiguous = row["is_ambiguous"]

        # Normalize ambiguity → bool
        if isinstance(is_ambiguous, str):
            is_amb_flag = is_ambiguous.lower() in ["true", "1", "yes"]
        else:
            is_amb_flag = bool(is_ambiguous)

        # -------------------------
        # CHOSEN: ground-truth behaviour
        # -------------------------
        chosen_action = "Clarify" if is_amb_flag else "Answer"

        chosen = build_structured_output(
            action=chosen_action,
            reasoning=reasoning,
            facets=facets,
            response=response,
        )

        # -------------------------
        # REJECTED: opposite action + manual template reasoning
        # -------------------------
        rejected_action = "Answer" if chosen_action == "Clarify" else "Clarify"

        if chosen_action == "Clarify":
            # Good = Clarify → Bad = Answer (answers directly, ignores ambiguity)
            rej_reasoning = (
                "This response answers directly without addressing the ambiguity "
                "or multiple possible interpretations in the question."
            )
            rej_facets = "[]"
        else:
            # Good = Answer → Bad = Clarify (asks for unnecessary clarification)
            rej_reasoning = (
                "This response asks for clarification even though the question "
                "is sufficiently clear to be answered directly."
            )
            rej_facets = '["Unnecessary Clarification"]'

        rejected = build_structured_output(
            action=rejected_action,
            reasoning=rej_reasoning,
            facets=rej_facets,
            response=neg_response,
        )

        # -------------------------
        # PROMPT: system + user (Qwen chat template style content)
        # -------------------------
        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
            f"<|im_start|>user\n{question}\n<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        obj = {
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
            "question": question,
            "true_action": chosen_action,
        }
        f.write(json.dumps(obj) + "\n")

print("Saved synthetic DPO dataset to:", out_path)

100%|██████████| 10036/10036 [00:00<00:00, 19031.99it/s]

Saved synthetic DPO dataset to: dpo_dataset_clean.jsonl


In [10]:
import json, random

lines = open("dpo_dataset_clean.jsonl").read().splitlines()
for row in random.sample(lines, 5):
    obj = json.loads(row)
    print("Q:", obj["question"])
    print("\nCHOSEN:\n", obj["chosen"])
    print("\nREJECTED:\n", obj["rejected"])
    print("="*80, "\n")

Q: Which is the shortest day in the southern hemisphere?

CHOSEN:
 Action: Clarify
Reasoning: The question is ambiguous because while it asks for the shortest day, it does not specify a particular year or if it's asking for the general astronomical date which remains consistent annually. This lack of specificity about the date makes the answer potentially vary based on leap years or specific regional daylight saving adjustments. Question: Which is the shortest day in the southern hemisphere?
Facets: ['Type of Information Requested', 'Specificity of Date']
Reasoning: The question is ambiguous because it does not specify a particular year or whether it is asking for the general astronomical date, which remains consistent annually. This lack of
Facets: ["Type of Information Requested", "Specificity of Date"]
Response: Are you looking for the general concept or the specific date of the shortest day in the southern hemisphere?

REJECTED:
 Action: Answer
Reasoning: This response answers dire

# removing duplication of reasoning in chosen results

In [12]:
import json
import re

input_path = "dpo_dataset_clean.jsonl"
output_path = "dpo_dataset_clean_final.jsonl"

def clean_block(text):
    """
    Cleans duplicated Reasoning: and Facets: inside the CHOSEN block.
    - Keeps only the FIRST Reasoning: section
    - Keeps only the FIRST Facets: section
    - Removes duplicate embedded Reasoning/Facets text
    """
    
    # Extract the structured 4 fields
    lines = text.split("\n")
    
    action_line = None
    reasoning_line = None
    facets_line = None
    response_line = None
    leftovers = []

    # First pass — capture the FIRST Action / Reasoning / Facets / Response
    for line in lines:
        stripped = line.strip()

        if stripped.startswith("Action:") and action_line is None:
            action_line = stripped
        elif stripped.startswith("Reasoning:") and reasoning_line is None:
            reasoning_line = stripped
        elif stripped.startswith("Facets:") and facets_line is None:
            facets_line = stripped
        elif stripped.startswith("Response:") and response_line is None:
            response_line = stripped
        else:
            leftovers.append(stripped)

    # Remove any leftover lines that *contain* embedded "Reasoning:" or "Facets:"
    cleaned_leftovers = [
        line for line in leftovers
        if not (line.startswith("Reasoning:") or line.startswith("Facets:"))
    ]

    # Reconstruct clean block
    final = []
    if action_line: final.append(action_line)
    if reasoning_line: final.append(reasoning_line)
    if facets_line: final.append(facets_line)
    if response_line: final.append(response_line)

    # Ignore leftovers entirely (we do not need them for DPO)
    return "\n".join(final)

# Process the dataset
with open(input_path, "r") as infile, open(output_path, "w") as outfile:
    for line in infile:
        row = json.loads(line)

        # Clean chosen ONLY (rejected is already clean & short)
        row["chosen"] = clean_block(row["chosen"])

        outfile.write(json.dumps(row) + "\n")

print("✨ Cleaning complete! Saved to:", output_path)

✨ Cleaning complete! Saved to: dpo_dataset_clean_final.jsonl


In [13]:
import json, random

lines = open("dpo_dataset_clean_final.jsonl").read().splitlines()
for row in random.sample(lines, 5):
    obj = json.loads(row)
    print("Q:", obj["question"])
    print("\nCHOSEN:\n", obj["chosen"])
    print("\nREJECTED:\n", obj["rejected"])
    print("="*80, "\n")

Q: Who did the bears play in 1985 super bowl?

CHOSEN:
 Action: Answer
Reasoning: The question is ambiguous because it does not specify which team nicknamed "the Bears" is being referred to, as there could be multiple teams with that nickname across different sports or leagues. Additionally, the exact name of the opponent is not provided, making it unclear who they played against in the 1985 Super Bowl. Example 3 (Ambiguous):
Facets: []
Response: New England Patriots

REJECTED:
 Action: Clarify
Reasoning: This response asks for clarification even though the question is sufficiently clear to be answered directly.
Facets: ["Unnecessary Clarification"]
Response: Could you confirm if you're referring to the Super Bowl that the Chicago Bears played in?

Q: Who is the first president of bharatiya janata party?

CHOSEN:
 Action: Answer
Reasoning: The question is not ambiguous because it specifically asks for the first president of the Bharatiya Janata Party, which has a known historical figur

# Getting preference pairs from the policy model (doing this hybrid approach offline)

In [18]:
import json
import re

def extract_field(text, field_name):
    """
    Extracts a field like 'Action:' or 'Reasoning:' from a multi-line chosen string.
    Returns None if not found.
    """
    pattern = rf"{field_name}:(.*)"
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        return match.group(1).strip().lower()
    return None

def check_consistency(example):
    chosen = example["chosen"]
    question = example.get("question", "<no question>")
    
    # Extract fields from the CHOSEN block
    action = extract_field(chosen, "Action")
    reasoning = extract_field(chosen, "Reasoning")

    issues = []
    
    if action is None:
        issues.append("Missing Action")
    if reasoning is None:
        issues.append("Missing Reasoning")
        
    # Skip if missing essential fields
    if issues:
        print(f"⚠️ STRUCTURAL ISSUE in: {question} → {issues}")
        return False

    # Check semantic contradictions
    reasoning_lc = reasoning.lower()
    
    if "not ambiguous" in reasoning_lc and action == "clarify":
        print("❌ CONTRADICTION FOUND")
        print("Q:", question)
        print("Action:", action)
        print("Reasoning:", reasoning)
        print("---")
        return False
    
    if "ambiguous" in reasoning_lc and action == "answer":
        print("❌ CONTRADICTION FOUND")
        print("Q:", question)
        print("Action:", action)
        print("Reasoning:", reasoning)
        print("---")
        return False
    
    # No contradiction detected
    return True


# -------------------------
# RUN CHECKER ON JSONL FILE
# -------------------------

file_path = "dpo_dataset_clean_final.jsonl"
total = 0
bad = 0

with open(file_path, "r") as f:
    for line in f:
        total += 1
        example = json.loads(line)
        ok = check_consistency(example)
        if not ok:
            bad += 1

print("\n====================================")
print(f"Total examples checked: {total}")
print(f"Inconsistencies found: {bad}")
print("====================================\n")

❌ CONTRADICTION FOUND
Q: Who played george washington in the john adams series?
Action: answer
Reasoning: the question is not ambiguous because it specifies the character name (george washington) and the series title (john adams), providing sufficient context to identify the actor who portrayed george washington in that particular series. question: who played george washington in the john adams series?
---
❌ CONTRADICTION FOUND
Q: When did the frozen ride open at epcot?
Action: answer
Reasoning: the question is not ambiguous because it specifies the attraction name ("frozen ride") and the location (epcot), providing sufficient details to identify a specific opening date. question: when did the frozen ride open at epcot?
---
❌ CONTRADICTION FOUND
Q: Name the landforms that form the boundaries of the peninsular plateau?
Action: answer
Reasoning: the question is not ambiguous because it clearly asks for specific landforms that define the boundaries of a particular geographical feature, th

In [19]:
df_check=pd.read_csv('final_SFT_data_for_use.csv')
df_check

,Unnamed: 0,question,is_ambiguous,facets,reasoning,positive_response,action
0,0,When did the simpsons first air on television?,True,"['Type of Show', 'Platform']",The question is ambiguous because it does not ...,Are you asking about the original TV series or...,Clarify
1,1,Who played george washington in the john adams...,False,[],The question is not ambiguous because it speci...,['David Morse'],Answer
2,2,What is the legal age of marriage in usa?,True,"['State', 'Circumstances for Marriage']",The question is ambiguous because the legal ag...,In which U.S. state and under what circumstanc...,Clarify
3,3,Who starred in barefoot in the park on broadway?,True,['Character Name'],The question is ambiguous because it does not ...,Did you mean the character names or the actors...,Clarify
4,4,When did the manhattan project began and end?,True,['Time Frame Interpretation'],The question is ambiguous because it does not ...,In which specific year or time period are you ...,Clarify
...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,"['Geographical Region', 'Specific Location']",The question is ambiguous because it does not ...,In which geographical region and specific loca...,Clarify
10032,10032,Who is the band in the movie 10 things i hate ...,True,"['Location', 'Event Type']",The question is ambiguous because it does not ...,Are you asking about the band featured in the ...,Clarify
10033,10033,Who was the last person in the uk to be executed?,False,[],The question is not ambiguous because it speci...,['Gwynne Evans and Peter Allen'],Answer
10034,10034,Who does wonder woman end up with in the comics?,True,"['Story Universe', 'Publication Series']",The question is ambiguous because 'Wonder Woma...,Are you asking about Wonder Woman's relationsh...,Clarify


# From the above code we have realized that reasoning in the data is extremely inconsistent
# so we will be using template reasoning instead

In [1]:
import json
from tqdm import tqdm

INPUT_FILE = "dpo_dataset.jsonl"            # your existing file
OUTPUT_FILE = "dpo_dataset_cleaned.jsonl"   # new cleaned file

ANSWER_REASONING = (
    "The question is clear and has a single reasonable interpretation, "
    "so it can be answered directly."
)

CLARIFY_REASONING = (
    "The question can be interpreted in multiple ways or lacks specific details, "
    "so clarification is needed before providing an answer."
)


def normalize_reasoning_block(text):
    """
    text: full 'chosen' or 'rejected' string with:
      Action: ...
      Reasoning: ...
      Facets: ...
      Response: ...

    We:
      - detect Action (Clarify / Answer)
      - replace the FIRST Reasoning: line with a template
      - drop any additional Reasoning: lines
    """
    if not isinstance(text, str):
        return text  # unexpected, just return as-is

    lines = text.splitlines()

    # 1) detect action from the 'Action:' line
    action = None
    for l in lines:
        stripped = l.strip().lower()
        if stripped.startswith("action:"):
            val = l.split(":", 1)[1].strip().lower()
            if "clarify" in val:
                action = "Clarify"
            elif "answer" in val:
                action = "Answer"
            break

    # if we can't find action, just return unchanged
    if action is None:
        return text

    # 2) pick template based on action
    if action == "Answer":
        template = ANSWER_REASONING
    else:
        template = CLARIFY_REASONING

    # 3) rewrite Reasoning line(s)
    new_lines = []
    replaced = False
    for l in lines:
        stripped = l.strip().lower()
        if stripped.startswith("reasoning:"):
            if not replaced:
                # keep exactly one standardized reasoning
                new_lines.append("Reasoning: " + template)
                replaced = True
            else:
                # drop duplicate / trailing Reasoning lines
                continue
        else:
            new_lines.append(l)

    # if there was no Reasoning line at all, we can optionally insert one
    if not replaced:
        # insert before Facets if exists, else append at end
        inserted = False
        new2 = []
        for l in new_lines:
            if (not inserted) and l.strip().lower().startswith("facets:"):
                new2.append("Reasoning: " + template)
                inserted = True
            new2.append(l)
        if not inserted:
            new2.append("Reasoning: " + template)
        new_lines = new2

    return "\n".join(new_lines)


# ----------------- RUN CLEANING -----------------

total = 0
written = 0

with open(INPUT_FILE, "r") as f_in, open(OUTPUT_FILE, "w") as f_out:
    lines = f_in.readlines()
    print(f"Processing {len(lines)} examples...")

    for line in tqdm(lines):
        line = line.strip()
        if not line:
            continue

        try:
            ex = json.loads(line)
        except Exception as e:
            # malformed line – skip
            continue

        # we assume ex has keys: prompt, chosen, rejected, question, true_action, etc.
        if "chosen" in ex:
            ex["chosen"] = normalize_reasoning_block(ex["chosen"])
        if "rejected" in ex:
            ex["rejected"] = normalize_reasoning_block(ex["rejected"])

        f_out.write(json.dumps(ex) + "\n")
        written += 1
        total += 1

print("✅ Cleaning complete")
print("Total examples processed:", total)
print("Total examples written:", written)
print("Saved cleaned dataset to:", OUTPUT_FILE)

Processing 10036 examples...


100%|██████████| 10036/10036 [00:00<00:00, 48061.00it/s]

✅ Cleaning complete
Total examples processed: 10036
Total examples written: 10036
Saved cleaned dataset to: dpo_dataset_cleaned.jsonl


In [2]:
import json
from itertools import islice

with open("dpo_dataset_cleaned.jsonl") as f:
    for line in islice(f, 5):
        ex = json.loads(line)
        print("QUESTION:", ex.get("question"))
        print("CHOSEN:\n", ex["chosen"])
        print("REJECTED:\n", ex["rejected"])
        print("="*80)

QUESTION: When did the simpsons first air on television?
CHOSEN:
 Action: Clarify
Reasoning: The question can be interpreted in multiple ways or lacks specific details, so clarification is needed before providing an answer.
Facets: ['Format Type', 'Show Type']

Facets: ["Format Type", "Show Type"]
Response: Are you asking about the first air date of The Simpsons' pilot, or the start of its full series run on television?"?
REJECTED:
 Action: Answer
Reasoning: The question is clear and has a single reasonable interpretation, so it can be answered directly.
Facets: []
Response: The Simpsons first aired on December 17, 1989.
QUESTION: Who played george washington in the john adams series?
CHOSEN:
 Action: Answer
Reasoning: The question is clear and has a single reasonable interpretation, so it can be answered directly.
Facets: []

Facets: []
Response: David Morse
REJECTED:
 Action: Clarify
Reasoning: The question can be interpreted in multiple ways or lacks specific details, so clarificati

# facets are duplicated, removing this

In [3]:
import json
from tqdm import tqdm

INPUT_FILE = "dpo_dataset_cleaned.jsonl"
OUTPUT_FILE = "dpo_dataset_final.jsonl"

def polish_block(text):
    """
    Cleans:
    - duplicate Facets lines
    - stray Question: lines
    """
    if not isinstance(text, str):
        return text

    lines = text.splitlines()
    new_lines = []

    seen_facets = False

    for l in lines:
        stripped = l.strip()

        # Remove internal "Question:" lines
        if stripped.lower().startswith("question:"):
            continue

        # Allow only ONE Facets: line
        if stripped.lower().startswith("facets:"):
            if seen_facets:
                continue
            seen_facets = True
            new_lines.append(l)
            continue

        new_lines.append(l)

    return "\n".join(new_lines)


# Process file
total = 0
written = 0

with open(INPUT_FILE, "r") as f_in, open(OUTPUT_FILE, "w") as f_out:
    lines = f_in.readlines()

    print(f"Polishing {len(lines)} examples...")

    for line in tqdm(lines):
        line = line.strip()
        if not line:
            continue

        try:
            ex = json.loads(line)
        except:
            continue

        if "chosen" in ex:
            ex["chosen"] = polish_block(ex["chosen"])

        if "rejected" in ex:
            ex["rejected"] = polish_block(ex["rejected"])

        f_out.write(json.dumps(ex) + "\n")
        written += 1
        total += 1

print("✅ Final cleaning complete")
print("Total examples processed:", total)
print("Saved polished dataset to:", OUTPUT_FILE)

Polishing 10036 examples...


100%|██████████| 10036/10036 [00:00<00:00, 49652.02it/s]

✅ Final cleaning complete
Total examples processed: 10036
Saved polished dataset to: dpo_dataset_final.jsonl


In [4]:
import json
from itertools import islice

with open("dpo_dataset_final.jsonl") as f:
    for line in islice(f, 5):
        ex = json.loads(line)
        print("QUESTION:", ex.get("question"))
        print("CHOSEN:\n", ex["chosen"])
        print("REJECTED:\n", ex["rejected"])
        print("="*80)

QUESTION: When did the simpsons first air on television?
CHOSEN:
 Action: Clarify
Reasoning: The question can be interpreted in multiple ways or lacks specific details, so clarification is needed before providing an answer.
Facets: ['Format Type', 'Show Type']

Response: Are you asking about the first air date of The Simpsons' pilot, or the start of its full series run on television?"?
REJECTED:
 Action: Answer
Reasoning: The question is clear and has a single reasonable interpretation, so it can be answered directly.
Facets: []
Response: The Simpsons first aired on December 17, 1989.
QUESTION: Who played george washington in the john adams series?
CHOSEN:
 Action: Answer
Reasoning: The question is clear and has a single reasonable interpretation, so it can be answered directly.
Facets: []

Response: David Morse
REJECTED:
 Action: Clarify
Reasoning: The question can be interpreted in multiple ways or lacks specific details, so clarification is needed before providing an answer.
Facets:

# Offline hybrid mining of preference pairs

# sanity check for data

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_ADAPTER = "bhavanasanghi9/clarifier-sft-qwen-7b"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, LORA_ADAPTER)
model.eval()

# FIXED: Added few-shot examples in the EXACT training format
prompt = """<|im_start|>system
You are a helpful assistant that detects ambiguity in questions.
<|im_end|>
<|im_start|>user
Question: Who won the US Open?
<|im_end|>
<|im_start|>assistant
Action: Clarify
Facets: ['sport', 'year']
Reasoning: The question is ambiguous because "US Open" refers to tournaments in multiple sports (tennis, golf) and occurs annually, so both the sport and year need clarification.
Response: Which sport's US Open are you asking about, and which year?
<|im_end|>
<|im_start|>user
Question: What is the capital of France?
<|im_end|>
<|im_start|>assistant
Action: Answer
Facets: []
Reasoning: The question is clear and unambiguous - it asks for a well-established geographical fact with one definitive answer.
Response: The capital of France is Paris.
<|im_end|>
<|im_start|>user
Question: When was the first remote control TV invented?
<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

result = tokenizer.decode(output[0], skip_special_tokens=False)
print(result)

# Extract just the assistant's response
assistant_response = result.split("<|im_start|>assistant")[-1].split("<|im_end|>")[0].strip()
print("\n" + "="*70)
print("ASSISTANT RESPONSE:")
print("="*70)
print(assistant_response)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

<|im_start|>system
You are a helpful assistant that detects ambiguity in questions.
<|im_end|>
<|im_start|>user
Question: Who won the US Open?
<|im_end|>
<|im_start|>assistant
Action: Clarify
Facets: ['sport', 'year']
Reasoning: The question is ambiguous because "US Open" refers to tournaments in multiple sports (tennis, golf) and occurs annually, so both the sport and year need clarification.
Response: Which sport's US Open are you asking about, and which year?
<|im_end|>
<|im_start|>user
Question: What is the capital of France?
<|im_end|>
<|im_start|>assistant
Action: Answer
Facets: []
Reasoning: The question is clear and unambiguous - it asks for a well-established geographical fact with one definitive answer.
Response: The capital of France is Paris.
<|im_end|>
<|im_start|>user
Question: When was the first remote control TV invented?
<|im_end|>
<|im_start|>assistant
Action: Answer
Facets: []
Reasoning: The question is not ambiguous because it clearly asks for a specific historical 

In [14]:
# Load your training dataset
from datasets import load_dataset

# OR if you have a local file
import json
with open('sft_data.jsonl', 'r') as f:
    first_example = json.loads(f.readline())
    print(first_example)

{'messages': [{'role': 'system', 'content': 'You are a question-understanding agent. Your job is to detect ambiguity, explain your reasoning, extract facets, and either ask a clarifying question (if ambiguous) or answer directly (if unambiguous). Always follow this exact output format:\n\nAction: Clarify|Answer\nReasoning: <reasoning>\nFacets: <json list>\nResponse: <clarifying question or answer>\n'}, {'role': 'user', 'content': 'When did the simpsons first air on television?'}, {'role': 'assistant', 'content': 'Action: Clarify\nReasoning: The question is ambiguous because it does not specify whether the inquiry is about the first episode, the first season, or a specific format type such as the prime-time animated series. Additionally, the term "Show Type" could refer to different formats like the original series versus movies or specials, which are also part of The Simpsons media franchise. These unspecified facets lead to potential multiple interpretations. Question: When did the si

In [10]:
import json

with open("dpo_dataset_cleaned.jsonl") as f:
    for _ in range(3):
        line = json.loads(next(f))
        print("TYPE of chosen:", type(line["chosen"]))
        print("RAW chosen:")
        print(line["chosen"])
        print("----------")

TYPE of chosen: <class 'str'>
RAW chosen:
Action: Clarify
Reasoning: The question can be interpreted in multiple ways or lacks specific details, so clarification is needed before providing an answer.
Facets: ['Format Type', 'Show Type']

Facets: ["Format Type", "Show Type"]
Response: Are you asking about the first air date of The Simpsons' pilot, or the start of its full series run on television?"?
----------
TYPE of chosen: <class 'str'>
RAW chosen:
Action: Answer
Reasoning: The question is clear and has a single reasonable interpretation, so it can be answered directly.
Facets: []

Facets: []
Response: David Morse
----------
TYPE of chosen: <class 'str'>
RAW chosen:
Action: Clarify
Reasoning: The question can be interpreted in multiple ways or lacks specific details, so clarification is needed before providing an answer.
Facets: ['State Specificity', 'Legal Circumstances']
Facets: ["State Specificity", "Legal Circumstances"]
Response: Are you asking about a specific state and under w

# Mining begin

In [6]:
!pip install --user numpy

In [4]:
import json
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import re

# =========================
# CONFIG
# =========================
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_ADAPTER = "bhavanasanghi9/clarifier-sft-qwen-7b"

SFT_DATA_PATH = "sft_data.jsonl"          # your SFT training data
OUTPUT_PATH = "hybrid_dpo_from_sft.jsonl" # mined hybrid DPO pairs

MAX_EXAMPLES = 2000    # <- change if you want more/less mining
BATCH_SIZE = 8         # batches for faster generation
MAX_NEW_TOKENS = 200   # keep modest for speed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# Load model + tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
model.eval()

print("✅ Model loaded.")


# =========================
# Helper: extract Action and Response from model output
# =========================
def extract_action(text: str):
    text = text.strip()
    for line in text.split("\n"):
        if line.lower().startswith("action:"):
            val = line.split(":", 1)[1].strip()
            # Normalize capitalization
            if val.lower().startswith("clarify"):
                return "Clarify"
            if val.lower().startswith("answer"):
                return "Answer"
    return None  # couldn't parse


def extract_response(text: str):
    text = text.strip()
    for line in text.split("\n"):
        if line.lower().startswith("response:"):
            val = line.split(":", 1)[1].strip()
            return val
    return None


# =========================
# Helper: build prompt
# We will use the SAME style you just tested successfully
# Few-shots are baked into the prompt string; we just plug in the new question.
# =========================
FEWSHOT_PREFIX = """<|im_start|>system
You are a helpful assistant that detects ambiguity in questions.
<|im_end|>
<|im_start|>user
Question: Who won the US Open?
<|im_end|>
<|im_start|>assistant
Action: Clarify
Facets: ['sport', 'year']
Reasoning: The question is ambiguous because "US Open" refers to tournaments in multiple sports (tennis, golf) and occurs annually, so both the sport and year need clarification.
Response: Which sport's US Open are you asking about, and which year?
<|im_end|>
<|im_start|>user
Question: What is the capital of France?
<|im_end|>
<|im_start|>assistant
Action: Answer
Facets: []
Reasoning: The question is clear and unambiguous - it asks for a well-established geographical fact with one definitive answer.
Response: The capital of France is Paris.
<|im_end|>
"""

def build_prompt(question: str) -> str:
    """
    Use few-shot prefix + new question in the exact format you tested.
    """
    prompt = FEWSHOT_PREFIX + f"<|im_start|>user\nQuestion: {question}\n<|im_end|>\n<|im_start|>assistant\n"
    return prompt


# =========================
# Mining loop (batched)
# =========================
def mine_hybrid_pairs():
    total_seen = 0
    total_pairs = 0

    # We'll stream input and process in batches
    batch_questions = []
    batch_gold_assistant = []
    batch_prompts = []

    with open(SFT_DATA_PATH, "r") as f_in, open(OUTPUT_PATH, "w") as f_out:
        # We'll wrap reading lines in tqdm for progress
        for line in tqdm(f_in, total=MAX_EXAMPLES, desc="Mining from SFT data"):
            if total_seen >= MAX_EXAMPLES:
                break

            example = json.loads(line)
            messages = example["messages"]

            # Expect: [system, user, assistant]
            system_msg = messages[0]["content"]
            user_msg = messages[1]["content"]  # should be the question text
            assistant_msg = messages[2]["content"]  # gold structured output

            # Get gold action from the assistant message
            gold_action = None
            for al in assistant_msg.split("\n"):
                if al.lower().startswith("action:"):
                    v = al.split(":", 1)[1].strip()
                    if v.lower().startswith("clarify"):
                        gold_action = "Clarify"
                    elif v.lower().startswith("answer"):
                        gold_action = "Answer"
                    break

            if gold_action is None:
                # skip weird rows
                total_seen += 1
                continue

            # Build prompt using FEWSHOT_PREFIX (ignoring old system for consistency)
            question_text = user_msg.strip()
            prompt = build_prompt(question_text)

            batch_questions.append(question_text)
            batch_gold_assistant.append(assistant_msg)
            batch_prompts.append(prompt)
            total_seen += 1

            # If batch is full or we've hit MAX_EXAMPLES, run generation
            if len(batch_prompts) == BATCH_SIZE or total_seen >= MAX_EXAMPLES:
                # Tokenize batch
                inputs = tokenizer(
                    batch_prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False,
                        eos_token_id=tokenizer.eos_token_id,
                    )

                decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)

                # For each output, parse and compare actions
                for q_text, gold_ass, model_raw, prompt_str in zip(
                    batch_questions, batch_gold_assistant, decoded, batch_prompts
                ):
                    # Extract only the last assistant segment
                    if "<|im_start|>assistant" in model_raw:
                        model_segment = model_raw.split("<|im_start|>assistant")[-1]
                        # Make sure to stop at the next <|im_end|> to avoid trailing stuff
                        if "<|im_end|>" in model_segment:
                            model_segment = model_segment.split("<|im_end|>")[0].strip()
                        model_resp = model_segment.strip()
                    else:
                        model_resp = model_raw.strip()

                    pred_action = extract_action(model_resp)

                    # Get gold action again for this example
                    gold_action_local = None
                    for al in gold_ass.split("\n"):
                        if al.lower().startswith("action:"):
                            v = al.split(":", 1)[1].strip()
                            if v.lower().startswith("clarify"):
                                gold_action_local = "Clarify"
                            elif v.lower().startswith("answer"):
                                gold_action_local = "Answer"
                            break

                    # If we couldn't parse predicted action, skip
                    if pred_action is None or gold_action_local is None:
                        continue

                    # Only create a DPO pair when actions disagree
                    if pred_action != gold_action_local:
                        dpo_example = {
                            "prompt": prompt_str,      # full system+fewshot+user
                            "chosen": gold_ass,        # gold assistant content
                            "rejected": model_resp,    # SFT model's mistaken response
                            "question": q_text,
                            "gold_action": gold_action_local,
                            "pred_action": pred_action,
                        }
                        f_out.write(json.dumps(dpo_example) + "\n")
                        total_pairs += 1

                # Clear batch
                batch_questions = []
                batch_gold_assistant = []
                batch_prompts = []

                # Optional: print some lightweight logging
                tqdm.write(f"Processed {total_seen} examples, mined {total_pairs} pairs so far.")

    print("🏁 Mining finished.")
    print(f"Total SFT examples seen: {total_seen}")
    print(f"Total hybrid DPO pairs mined: {total_pairs}")


mine_hybrid_pairs()

RuntimeError: Failed to import transformers.models.auto.tokenization_auto because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):
numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [3]:
import transformers
print(transformers.__version__)

4.48.0


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import time

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_ADAPTER = "bhavanasanghi9/clarifier-sft-qwen-7b"
MERGED_MODEL = "./clarifier_sft_merged"

def merge_model():
    """Merge LoRA adapter with base model"""
    total_start = time.time()
    
    # Step 1: Load base model
    print("=" * 70)
    print("⏳ STEP 1/5: Loading base model...")
    print("=" * 70)
    start = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        load_in_8bit=True,
        device_map="auto",  # Use GPU if available
        trust_remote_code=True
    )
    print(f"✅ Base model loaded in {time.time() - start:.1f}s\n")
    
    # Step 2: Load LoRA adapter
    print("=" * 70)
    print("⏳ STEP 2/5: Loading LoRA adapter...")
    print("=" * 70)
    start = time.time()
    model = PeftModel.from_pretrained(model, LORA_ADAPTER)
    print(f"✅ LoRA adapter loaded in {time.time() - start:.1f}s\n")
    
    # Step 3: Merge weights
    print("=" * 70)
    print("⏳ STEP 3/5: Merging weights (this is the slowest step)...")
    print("=" * 70)
    start = time.time()
    merged = model.merge_and_unload()
    print(f"✅ Weights merged in {time.time() - start:.1f}s\n")
    
    # Step 4: Save model
    print("=" * 70)
    print("⏳ STEP 4/5: Saving merged model...")
    print("=" * 70)
    start = time.time()
    merged.save_pretrained(MERGED_MODEL)
    print(f"✅ Model saved in {time.time() - start:.1f}s\n")
    
    # Step 5: Save tokenizer
    print("=" * 70)
    print("⏳ STEP 5/5: Saving tokenizer...")
    print("=" * 70)
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.save_pretrained(MERGED_MODEL)
    print(f"✅ Tokenizer saved in {time.time() - start:.1f}s\n")
    
    # Summary
    total_time = time.time() - total_start
    print("=" * 70)
    print(f"🎉 COMPLETE! Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
    print(f"📁 Merged model saved to: {MERGED_MODEL}")
    print("=" * 70)

if __name__ == "__main__":
    merge_model()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


⏳ STEP 1/5: Loading base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Base model loaded in 65.1s

⏳ STEP 2/5: Loading LoRA adapter...


/opt/conda/lib/python3.11/site-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version', 'qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


✅ LoRA adapter loaded in 12.7s

⏳ STEP 3/5: Merging weights (this is the slowest step)...


/opt/conda/lib/python3.11/site-packages/peft/tuners/lora/bnb.py:85: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


✅ Weights merged in 56.0s

⏳ STEP 4/5: Saving merged model...
✅ Model saved in 32.1s

⏳ STEP 5/5: Saving tokenizer...
✅ Tokenizer saved in 0.7s

🎉 COMPLETE! Total time: 166.6s (2.8 minutes)
📁 Merged model saved to: ./clarifier_sft_merged


In [10]:
import torch
torch.__version__

'2.4.1+cu121'

# DPO PIPELINE

# using bhoomika's DPO code

In [1]:
import torch
print(f"GPU RAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU RAM used: 0.00 GB


In [2]:
# Cell: After restart - verify TRL
import sys
sys.path.insert(0, '/home/bsanghi2/.local/lib/python3.11/site-packages')

import trl
print(f"TRL version: {trl.__version__}")
print(f"TRL location: {trl.__file__}")

# Try importing DPOTrainer
from trl import DPOTrainer
print("✅ DPOTrainer imported successfully!")

TRL version: 0.9.6
TRL location: /home/bsanghi2/.local/lib/python3.11/site-packages/trl/__init__.py
✅ DPOTrainer imported successfully!


In [3]:
import pandas as pd
import numpy as np
dpo_df=pd.read_csv('regen_dpo_dataset.csv')
dpo_df['action']=np.where(dpo_df['is_ambiguous']==True,'Clarify', 'Answer')
dpo_df

,Unnamed: 0,question,is_ambiguous,facets_clean,response,reasoning,neg_response,action
0,0,When did the simpsons first air on television?,True,"[""Format Type"", ""Show Type""]",Are you asking about the first air date of The...,The question is ambiguous because it does not ...,"The Simpsons first aired on December 17, 1989.",Clarify
1,1,Who played george washington in the john adams...,False,[],David Morse,The question is not ambiguous because it speci...,Could you tell me which season of the John Ada...,Answer
2,2,What is the legal age of marriage in usa?,True,"[""State Specificity"", ""Legal Circumstances""]",Are you asking about a specific state and unde...,"The question is ambiguous because the ""legal a...",The legal age of marriage in the USA is 18.,Clarify
3,3,Who starred in barefoot in the park on broadway?,True,"[""Character Role""]",Are you asking about the actors who played spe...,The question is ambiguous because while it spe...,Walter Matthau and Anne Bancroft starred in Ba...,Clarify
4,4,When did the manhattan project began and end?,True,"[""Definition of Project Phases""]",Are you asking about the start and end dates o...,The question is ambiguous because it does not ...,The Manhattan Project began in 1942 and ended ...,Clarify
...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,True,"[""Location"", ""Geographical Region""]",In which location or geographical region are y...,The question is ambiguous because it does not ...,Summer holidays start in June.,Clarify
10032,10032,Who is the band in the movie 10 things i hate ...,True,"[""Event Location"", ""Action""]",Are you asking about the band playing at a spe...,The question is not ambiguous because it speci...,No Doubt.,Clarify
10033,10033,Who was the last person in the uk to be executed?,False,[],Gwynne Evans and Peter Allen,The question is ambiguous because it does not ...,Could you specify which method of execution yo...,Answer
10034,10034,Who does wonder woman end up with in the comics?,True,"[""Publication Series""]",Which publication series of Wonder Woman are y...,The question is not ambiguous because it speci...,Wonder Woman ends up with Steve Trevor in the ...,Clarify


In [7]:
# Cell: Clean reinstall TRL
import sys
import subprocess

print("Step 1: Uninstalling TRL...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "trl", "-y"])

print("\nStep 2: Clearing cache...")
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"])

print("\nStep 3: Installing TRL 0.9.6...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--user", "--no-cache-dir", "trl==0.9.6"],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("\n✅ Clean install complete!")
print("⚠️ RESTART YOUR KERNEL NOW")

Step 1: Uninstalling TRL...
Found existing installation: trl 0.9.6
Uninstalling trl-0.9.6:
  Successfully uninstalled trl-0.9.6

Step 2: Clearing cache...
Files removed: 24

Step 3: Installing TRL 0.9.6...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/245.8 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 20.1 MB/s eta 0:00:00

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


✅ Clean install complete!
⚠️ RESTART YOUR KERNEL NOW


In [1]:
!pip install transformers==4.48.0

In [2]:
!pip install --user transformers==4.48.0

In [6]:
!pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 83.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.4.1 requires fsspec[http]<=2025.10.0,>=2023.1.0, but you have fsspec 2025.12.0 which is incompatible.
torchvision 0.19.0 requires torch==2.4.0, but you have torch 2.9.1 which is incompatible.
vllm 0.6.3.post1 requires torch==2.4.0, but you have torch 2.9.1 which is incompatible.
xformers 0.0.27.post2 requires torch==2.4.0, but you have torch 2.9.1 which is incompatible.


In [7]:
!pip install --user numpy==1.26.4

In [ ]:
!pip install --user torch==2.3.1+cu121 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 4.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 60.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 23.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 75.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 4.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 7.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 24.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 44.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 21.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 15.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip install peft==0.15.0

In [4]:
import sys, ast, time
import torch
import pandas as pd
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import PeftModel
from trl import DPOTrainer
print(trl.__version__)

0.9.6
